In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    brier_score_loss
)

In [ ]:
# ============================================================
# RUNTIME BENCHMARK LOGGER
# ============================================================

from time import perf_counter
from datetime import datetime
from pathlib import Path
import pandas as pd
import os

RUNTIME_FILE = Path("FINAL_experimental_runtime_summary.csv")

if RUNTIME_FILE.exists():
    RUNTIME_LOG = pd.read_csv(RUNTIME_FILE).to_dict("records")
else:
    RUNTIME_LOG = []


def start_timer(experiment_name):
    print("\n" + "=" * 75)
    print(f"STARTING: {experiment_name}")
    print("Clock time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("=" * 75)

    return perf_counter()


def stop_timer(experiment_name, start_time):
    elapsed_sec = perf_counter() - start_time

    record = {
        "experiment": experiment_name,
        "runtime_seconds": elapsed_sec,
        "runtime_minutes": elapsed_sec / 60.0,
        "runtime_hours": elapsed_sec / 3600.0,
    }

    RUNTIME_LOG.append(record)

    pd.DataFrame(RUNTIME_LOG).to_csv(
        RUNTIME_FILE,
        index=False
    )

    print("\n" + "-" * 75)
    print(f"FINISHED: {experiment_name}")
    print(f"Runtime: {elapsed_sec:.2f} seconds")
    print(f"Runtime: {elapsed_sec / 60.0:.4f} minutes")
    print("-" * 75)

    return elapsed_sec

In [ ]:
df = pd.read_csv("../Data/Raw/triage_cleaned_working.csv")

registry = pd.read_csv("../Data/Raw/approved_slice_registry_FINAL.csv")

# Keep only accepted slices
registry = registry[registry["final_accept"] == "yes"].copy()

print("Accepted slices:", len(registry))

In [ ]:
def build_slice_membership(df, registry):
    df = df.copy()

    # Derived simple slice columns
    df["arrival_mode_ambulance"] = (
        df["arrival_mode"].astype(str).str.lower() == "ambulance"
    ).astype(int)

    df["arrival_mode_wheelchair"] = (
        df["arrival_mode"].astype(str).str.lower() == "wheelchair"
    ).astype(int)

    df["arrival_mode_unknown"] = (
        df["arrival_mode"].astype(str).str.lower() == "unknown"
    ).astype(int)

    df["tachycardia_flag"] = (
        df["heart_rate_final"].notna() & (df["heart_rate_final"] > 100)
    ).astype(int)

    df["hypotension_flag"] = (
        df["bp_systolic"].notna() & (df["bp_systolic"] < 90)
    ).astype(int)

    df["tachypnea_flag"] = (
        df["respiratory"].notna() & (df["respiratory"] > 20)
    ).astype(int)

    df["hypoxia_flag"] = (
        df["spo2"].notna() & (df["spo2"] < 95)
    ).astype(int)

    df["fever_temp_flag"] = (
        df["temperature"].notna() & (df["temperature"] >= 38.0)
    ).astype(int)

    df["hypothermia_flag"] = (
        df["temperature"].notna() & (df["temperature"] < 36.0)
    ).astype(int)

    # Full slice-rule dictionary
    slice_rules = {
        "chest_pain": lambda d: d["chest_pain"] == 1,
        "abdominal_pain": lambda d: d["abdominal_pain"] == 1,
        "back_pain": lambda d: d["back_pain"] == 1,
        "headache": lambda d: d["headache"] == 1,
        "cough": lambda d: d["cough"] == 1,
        "unclassified": lambda d: d["unclassified"] == 1,

        "ckd": lambda d: d["ckd"] == 1,
        "dm": lambda d: d["dm"] == 1,
        "htn": lambda d: d["htn"] == 1,
        "chd": lambda d: d["chd"] == 1,
        "pneumonia": lambda d: d["pneumonia"] == 1,

        "arrival_mode_ambulance": lambda d: d["arrival_mode_ambulance"] == 1,
        "arrival_mode_wheelchair": lambda d: d["arrival_mode_wheelchair"] == 1,
        "arrival_mode_unknown": lambda d: d["arrival_mode_unknown"] == 1,

        "vitals_missing_ge_3": lambda d: d["vitals_missing_count"] >= 3,
        "spo2_missing": lambda d: d["spo2_missing"] == 1,
        "hr_missing": lambda d: d["hr_missing"] == 1,

        "tachycardia": lambda d: d["tachycardia_flag"] == 1,
        "hypotension": lambda d: d["hypotension_flag"] == 1,
        "tachypnea": lambda d: d["tachypnea_flag"] == 1,
        "hypoxia": lambda d: d["hypoxia_flag"] == 1,
        "fever_temp": lambda d: d["fever_temp_flag"] == 1,
        "hypothermia": lambda d: d["hypothermia_flag"] == 1,

        "chest_pain_and_ambulance": lambda d: (
            (d["chest_pain"] == 1) & (d["arrival_mode_ambulance"] == 1)
        ),

        "abdominal_pain_and_hr_missing": lambda d: (
            (d["abdominal_pain"] == 1) & (d["hr_missing"] == 1)
        ),

        "ckd_and_spo2_missing": lambda d: (
            (d["ckd"] == 1) & (d["spo2_missing"] == 1)
        ),

        "pneumonia_and_tachypnea": lambda d: (
            (d["pneumonia"] == 1) & (d["tachypnea_flag"] == 1)
        ),

        "unclassified_and_vitals_missing_ge_3": lambda d: (
            (d["unclassified"] == 1) & (d["vitals_missing_count"] >= 3)
        ),

        "err_vitals_missing_ge_3": lambda d: d["vitals_missing_count"] >= 3,

        "err_spo2_missing_and_unclassified": lambda d: (
            (d["spo2_missing"] == 1) & (d["unclassified"] == 1)
        ),

        "err_hr_missing_and_abdominal_pain": lambda d: (
            (d["hr_missing"] == 1) & (d["abdominal_pain"] == 1)
        ),

        "err_pneumonia_and_tachypnea": lambda d: (
            (d["pneumonia"] == 1) & (d["tachypnea_flag"] == 1)
        ),

        "err_ckd_and_spo2_missing": lambda d: (
            (d["ckd"] == 1) & (d["spo2_missing"] == 1)
        ),

        "err_chest_pain_and_tachycardia": lambda d: (
            (d["chest_pain"] == 1) & (d["tachycardia_flag"] == 1)
        ),

        "err_unclassified_and_missingness": lambda d: (
            (d["unclassified"] == 1) & (d["vitals_missing_count"] >= 2)
        ),

        "err_hypoxia_and_arrival_missing": lambda d: (
            (d["hypoxia_flag"] == 1) & (d["arrival_mode_missing"] == 1)
        ),
    }

    membership = {}

    for _, row in registry.iterrows():
        name = row["slice_name"]

        if name in slice_rules:
            membership[name] = slice_rules[name](df).astype(int).values
        elif name in df.columns:
            membership[name] = df[name].astype(int).values
        else:
            print(f"Skipping unknown slice rule: {name}")

    return pd.DataFrame(membership, index=df.index)

Recreate slice membership matrix

In [ ]:
#ef add_derived_slice_columns(df):
#   df = df.copy()
#
#   # Arrival-mode derived columns
#   arrival = df["arrival_mode"].astype(str).str.lower().str.strip()
#
#   df["arrival_mode_ambulance"] = arrival.eq("ambulance").astype(int)
#   df["arrival_mode_wheelchair"] = arrival.eq("wheelchair").astype(int)
#   df["arrival_mode_unknown"] = arrival.eq("unknown").astype(int)
#
#   # Physiology derived columns
#   df["tachycardia_flag"] = (
#       df["heart_rate_final"].notna() & (df["heart_rate_final"] > 100)
#   ).astype(int)
#
#   df["hypotension_flag"] = (
#       df["bp_systolic"].notna() & (df["bp_systolic"] < 90)
#   ).astype(int)
#
#   df["tachypnea_flag"] = (
#       df["respiratory"].notna() & (df["respiratory"] > 20)
#   ).astype(int)
#
#   df["hypoxia_flag"] = (
#       df["spo2"].notna() & (df["spo2"] < 95)
#   ).astype(int)
#
#   df["fever_temp_flag"] = (
#       df["temperature"].notna() & (df["temperature"] >= 38.0)
#   ).astype(int)
#
#   df["hypothermia_flag"] = (
#       df["temperature"].notna() & (df["temperature"] < 36.0)
#   ).astype(int)
#
#   return df

In [ ]:
#ef build_slice_membership(df, registry):
#   df = add_derived_slice_columns(df)
#
#   slice_rules = {
#       # Single complaint slices
#       "chest_pain": lambda d: d["chest_pain"] == 1,
#       "abdominal_pain": lambda d: d["abdominal_pain"] == 1,
#       "back_pain": lambda d: d["back_pain"] == 1,
#       "headache": lambda d: d["headache"] == 1,
#       "cough": lambda d: d["cough"] == 1,
#       "unclassified": lambda d: d["unclassified"] == 1,
#
#       # Comorbidity slices
#       "ckd": lambda d: d["ckd"] == 1,
#       "dm": lambda d: d["dm"] == 1,
#       "htn": lambda d: d["htn"] == 1,
#       "chd": lambda d: d["chd"] == 1,
#       "pneumonia": lambda d: d["pneumonia"] == 1,
#
#       # Operational slices
#       "arrival_mode_ambulance": lambda d: d["arrival_mode_ambulance"] == 1,
#       "arrival_mode_wheelchair": lambda d: d["arrival_mode_wheelchair"] == 1,
#       "arrival_mode_unknown": lambda d: d["arrival_mode_unknown"] == 1,
#       "vitals_missing_ge_3": lambda d: d["vitals_missing_count"] >= 3,
#       "spo2_missing": lambda d: d["spo2_missing"] == 1,
#       "hr_missing": lambda d: d["hr_missing"] == 1,
#
#       # Physiology slices
#       "tachycardia": lambda d: d["tachycardia_flag"] == 1,
#       "hypotension": lambda d: d["hypotension_flag"] == 1,
#       "tachypnea": lambda d: d["tachypnea_flag"] == 1,
#       "hypoxia": lambda d: d["hypoxia_flag"] == 1,
#       "fever_temp": lambda d: d["fever_temp_flag"] == 1,
#       "hypothermia": lambda d: d["hypothermia_flag"] == 1,
#
#       # Degree-2 clinical slices
#       "chest_pain_and_ambulance": lambda d: (
#           (d["chest_pain"] == 1) & (d["arrival_mode_ambulance"] == 1)
#       ),
#       "abdominal_pain_and_hr_missing": lambda d: (
#           (d["abdominal_pain"] == 1) & (d["hr_missing"] == 1)
#       ),
#       "ckd_and_spo2_missing": lambda d: (
#           (d["ckd"] == 1) & (d["spo2_missing"] == 1)
#       ),
#       "pneumonia_and_tachypnea": lambda d: (
#           (d["pneumonia"] == 1) & (d["tachypnea_flag"] == 1)
#       ),
#       "unclassified_and_vitals_missing_ge_3": lambda d: (
#           (d["unclassified"] == 1) & (d["vitals_missing_count"] >= 3)
#       ),
#
#       # Error-guided slices
#       "err_vitals_missing_ge_3": lambda d: d["vitals_missing_count"] >= 3,
#       "err_spo2_missing_and_unclassified": lambda d: (
#           (d["spo2_missing"] == 1) & (d["unclassified"] == 1)
#       ),
#       "err_hr_missing_and_abdominal_pain": lambda d: (
#           (d["hr_missing"] == 1) & (d["abdominal_pain"] == 1)
#       ),
#       "err_pneumonia_and_tachypnea": lambda d: (
#           (d["pneumonia"] == 1) & (d["tachypnea_flag"] == 1)
#       ),
#       "err_ckd_and_spo2_missing": lambda d: (
#           (d["ckd"] == 1) & (d["spo2_missing"] == 1)
#       ),
#       "err_chest_pain_and_tachycardia": lambda d: (
#           (d["chest_pain"] == 1) & (d["tachycardia_flag"] == 1)
#       ),
#       "err_unclassified_and_missingness": lambda d: (
#           (d["unclassified"] == 1) & (d["vitals_missing_count"] >= 2)
#       ),
#       "err_hypoxia_and_arrival_missing": lambda d: (
#           (d["hypoxia_flag"] == 1) & (d["arrival_mode_missing"] == 1)
#       ),
#   }
#
#   membership = {}
#   skipped = []
#
#   for _, row in registry.iterrows():
#       name = row["slice_name"]
#
#       if name in slice_rules:
#           membership[name] = slice_rules[name](df).astype(int)
#       elif name in df.columns:
#           membership[name] = df[name].astype(int)
#       else:
#           skipped.append(name)
#
#   if skipped:
#       print("Skipped unknown slice names:")
#       for s in skipped:
#           print(" -", s)
#
#   return pd.DataFrame(membership, index=df.index)

In [ ]:
slice_membership = build_slice_membership(df, registry)

print(slice_membership.shape)
slice_membership.head()

In [ ]:
#def build_slice_membership(df, registry):
#    membership = {}
#
#    for _, row in registry.iterrows():
#        name = row["slice_name"]
#
#        # ⚠️ Replace with your actual rule logic
#        membership[name] = df[name].values  # assuming binary columns
#
#    return pd.DataFrame(membership)
#
#slice_membership = build_slice_membership(df, registry)
#
#slice_membership.head()

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Target
y_col = "y"

numeric_cols = [
    "age", "n_admissions", "heart_rate_final", "bp_systolic", "bp_diastole",
    "respiratory", "temperature", "spo2", "vitals_missing_count",
    "abdominal_pain", "chest_pain", "back_pain", "unclassified",
    "ckd", "dm", "headache", "htn", "cough", "chd", "asthma", "bronchitis",
    "dizziness", "glaucoma", "hepatitis", "paralysis", "pneumonia", "sicklecell",
    "arm_pain", "breast_pain", "dental_pain", "diarrhea", "ear_pain", "eye_pain",
    "fever", "pelvic_pain", "arrival_mode_missing", "hr_missing",
    "pulse_rate_missing", "bp_systolic_missing", "bp_diastole_missing",
    "respiratory_missing", "temperature_missing", "spo2_missing"
]

categorical_cols = ["arrival_mode", "arrival_month", "arrival_day", "arrival_hour_bin"]
X_cols = numeric_cols + categorical_cols

missing = [c for c in X_cols + [y_col] if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns in df: {missing}")

X = df[X_cols].copy()
y = df[y_col].astype(int)

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("lr", LogisticRegression(max_iter=5000, solver="lbfgs"))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(df), dtype=float)

for fold, (tr, te) in enumerate(cv.split(X, y), 1):
    model.fit(X.iloc[tr], y.iloc[tr])
    oof[te] = model.predict_proba(X.iloc[te])[:, 1]
    print(f"Fold {fold} done")

df["oof_pred_proba"] = np.clip(oof, 1e-6, 1 - 1e-6)
df["p_oof"] = df["oof_pred_proba"]

p = df["oof_pred_proba"]
yy = df[y_col].astype(int)

df["oof_logloss"] = -(yy * np.log(p) + (1 - yy) * np.log(1 - p))
df["oof_pred_label_05"] = (p >= 0.5).astype(int)
df["oof_error_05"] = (df["oof_pred_label_05"] != yy).astype(int)
df["false_negative_05"] = ((yy == 1) & (df["oof_pred_label_05"] == 0)).astype(int)

df["high_conf_contradiction"] = (
    ((yy == 1) & (p < 0.2)) |
    ((yy == 0) & (p > 0.8))
).astype(int)

print("OOF diagnostics created.")
print(df[[
    "oof_pred_proba",
    "oof_logloss",
    "oof_pred_label_05",
    "oof_error_05",
    "false_negative_05",
    "high_conf_contradiction"
]].head())

Compute slice harm scores (CRITICAL)

In [ ]:
import numpy as np

def compute_slice_harm(df, slice_membership):
    harms = {}

    global_loss = df["oof_logloss"].mean()
    global_fn = df["false_negative_05"].mean()
    global_hcc = df["high_conf_contradiction"].mean()

    for col in slice_membership.columns:
        mask = slice_membership[col].astype(bool)

        if mask.sum() == 0:
            harms[col] = np.nan
            continue

        loss_lift = df.loc[mask, "oof_logloss"].mean() - global_loss
        fn_lift = df.loc[mask, "false_negative_05"].mean() - global_fn
        hcc_lift = df.loc[mask, "high_conf_contradiction"].mean() - global_hcc

        harms[col] = loss_lift + fn_lift + hcc_lift

    return pd.Series(harms)

# ---- RUN THIS ----
slice_harm = compute_slice_harm(df, slice_membership)

print("Top harmful slices:")
print(slice_harm.sort_values(ascending=False).head(10))

In [ ]:
# ============================================================
# EXPORT COMPONENT-LEVEL COMPOSITE HARM AUDIT
# H_k = ΔL_k + ΔFN_k + ΔHCC_k
# ============================================================

harm_component_rows = []

global_loss = float(
    df["oof_logloss"].mean()
)

global_fn = float(
    df["false_negative_05"].mean()
)

global_hcc = float(
    df["high_conf_contradiction"].mean()
)

for slice_name in slice_membership.columns:

    mask = (
        slice_membership[slice_name]
        .fillna(0)
        .astype(bool)
    )

    support = int(mask.sum())

    if support == 0:
        continue

    loss_lift = float(
        df.loc[
            mask,
            "oof_logloss",
        ].mean()
        - global_loss
    )

    fn_lift = float(
        df.loc[
            mask,
            "false_negative_05",
        ].mean()
        - global_fn
    )

    hcc_lift = float(
        df.loc[
            mask,
            "high_conf_contradiction",
        ].mean()
        - global_hcc
    )

    composite_harm = (
        loss_lift
        + fn_lift
        + hcc_lift
    )

    harm_component_rows.append({
        "slice_name": slice_name,
        "support": support,
        "loss_lift": loss_lift,
        "fn_lift": fn_lift,
        "hcc_lift": hcc_lift,
        "composite_harm": composite_harm,
    })

slice_harm_components = (
    pd.DataFrame(
        harm_component_rows
    )
    .sort_values(
        "composite_harm",
        ascending=False,
    )
    .reset_index(drop=True)
)

# Confirm that the audit reproduces the active slice_harm.
audit_harm = (
    slice_harm_components
    .set_index("slice_name")[
        "composite_harm"
    ]
)

common_slices = (
    slice_harm.index
    .intersection(
        audit_harm.index
    )
)

if not np.allclose(
    slice_harm.loc[
        common_slices
    ].to_numpy(dtype=float),
    audit_harm.loc[
        common_slices
    ].to_numpy(dtype=float),
    equal_nan=True,
):
    raise ValueError(
        "The component audit does not reproduce slice_harm."
    )

display(
    slice_harm_components.head(15)
)

slice_harm_components.to_csv(
    "nb8_composite_harm_components.csv",
    index=False,
)

print(
    "Composite harm audit verified and saved."
)

In [ ]:
print("slice_harm exists:", "slice_harm" in globals())
print("slice_harm length:", len(slice_harm))
print(slice_harm.head())

In [ ]:
df["w_final"] = 1.0

positive_harm = slice_harm.dropna()
positive_harm = positive_harm[positive_harm > 0]

if len(positive_harm) > 0:
    positive_harm = positive_harm / positive_harm.max()

for s, h in positive_harm.items():
    if s in slice_membership.columns:
        df["w_final"] += h * slice_membership[s]

df["w_final"] = df["w_final"].clip(1.0, 5.0)

print(df["w_final"].describe())

In [ ]:
#def compute_slice_harm(df, slice_membership):
#    harms = {}
#
#    global_loss = df["oof_logloss"].mean()
#    global_fn = df["FN"].mean()
#    global_hcc = df["HCC"].mean()
#
#    for col in slice_membership.columns:
#        mask = slice_membership[col] == 1
#
#        if mask.sum() < 50:
#            continue
#
#        loss_lift = df.loc[mask, "oof_logloss"].mean() - global_loss
#        fn_lift = df.loc[mask, "FN"].mean() - global_fn
#        hcc_lift = df.loc[mask, "HCC"].mean() - global_hcc
#
#        # Combine (you can tune weights)
#        harm = (
#            0.5 * loss_lift +
#            0.3 * fn_lift +
#            0.2 * hcc_lift
#        )
#
#        harms[col] = harm
#
#    return pd.Series(harms)
#
#slice_harm = compute_slice_harm(df, slice_membership)
#
#slice_harm.sort_values(ascending=False).head(10)

In [ ]:
slice_harm = compute_slice_harm(df, slice_membership)
slice_harm.sort_values(ascending=False).head(10)

Normalize harm

In [ ]:
harm_norm = (slice_harm - slice_harm.min()) / (slice_harm.max() - slice_harm.min())

harm_norm = harm_norm.fillna(0)

harm_norm.head()

### **************************************************Dual-Source Testing*****************************************************

slice source labels

In [ ]:
### ============================================================
# STEP 4 ONWARD: PROVENANCE-AWARE DUAL-SOURCE ABLATION
# Clinical discovery vs Error-guided discovery vs Combined discovery
# ============================================================

import os
import numpy as np
import pandas as pd

if not isinstance(slice_harm, pd.Series):
    raise TypeError("slice_harm must remain a pandas Series. Re-run the harm computation cell before this.")

def read_first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return pd.read_csv(p), p
    raise FileNotFoundError(f"None found: {paths}")

full_registry, registry_path = read_first_existing([
    "approved_slice_registry_FINAL.csv",
    "../Data/Raw/approved_slice_registry_FINAL.csv",
])

clinical_candidates, clinical_path = read_first_existing([
    "clinical_slice_candidates.csv",
    "../Data/Raw/clinical_slice_candidates.csv",
])

error_candidates, error_path = read_first_existing([
    "error_guided_slice_candidates.csv",
    "../Data/Raw/error_guided_slice_candidates.csv",
])

print("Registry:", registry_path, full_registry.shape)
print("Clinical candidates:", clinical_path, clinical_candidates.shape)
print("Error-guided candidates:", error_path, error_candidates.shape)

# ------------------------------------------------------------
# 1. Keep accepted slices that exist in slice_membership
# ------------------------------------------------------------

# Use all registry rows for discovery-source ablation.
# Repair eligibility will be filtered later.
accepted_registry = full_registry.copy()
accepted_registry["final_accept_clean"] = accepted_registry["final_accept"].astype(str).str.lower().str.strip()

accepted_registry = accepted_registry[
    accepted_registry["slice_name"].isin(slice_membership.columns)
].copy()

print("Registry slices available in membership:", len(accepted_registry))

accepted_names = set(accepted_registry["slice_name"].astype(str))

print("Accepted slices available in membership:", len(accepted_names))

# ------------------------------------------------------------
# 2. Canonical mapping from err_ names to accepted clinical names
# ------------------------------------------------------------

error_to_canonical_alias = {
    "err_hr_missing_and_abdominal_pain": "abdominal_pain_and_hr_missing",
    "err_ckd_and_spo2_missing": "ckd_and_spo2_missing",
    "err_spo2_missing_and_unclassified": "spo2_missing_and_unclassified",
    "err_unclassified_and_missingness": "unclassified_and_vitals_missing_ge_3",
    "err_vitals_missing_ge_3": "vitals_missing_ge_3",
}

def canonicalize_error_slice(name):
    name = str(name)
    return error_to_canonical_alias.get(name, name)

clinical_raw = set(clinical_candidates["slice_name"].dropna().astype(str))
error_raw = set(error_candidates["slice_name"].dropna().astype(str))

clinical_supported = accepted_names.intersection(clinical_raw)

error_canonical_all = {
    canonicalize_error_slice(s) for s in error_raw
}

error_supported = accepted_names.intersection(error_canonical_all)

# Accepted operational/clinical/mixed slices not in clinical candidate file
# should still be counted as clinical-supported unless they are pure error-only.
for s in accepted_names:
    stype = ""
    if "slice_type_manual" in accepted_registry.columns:
        stype = str(
            accepted_registry.loc[
                accepted_registry["slice_name"] == s,
                "slice_type_manual"
            ].iloc[0]
        ).lower()

    if any(x in stype for x in ["clinical", "operational", "mixed"]):
        clinical_supported.add(s)

dual_supported = clinical_supported.intersection(error_supported)

# ------------------------------------------------------------
# 3. Build audit table
# ------------------------------------------------------------

source_audit = accepted_registry.copy()

source_audit["nb8_harm"] = source_audit["slice_name"].map(slice_harm)
source_audit["has_clinical_evidence"] = source_audit["slice_name"].isin(clinical_supported)
source_audit["has_error_guided_evidence"] = source_audit["slice_name"].isin(error_supported)
source_audit["has_dual_evidence"] = source_audit["slice_name"].isin(dual_supported)

def evidence_label(row):
    if row["has_clinical_evidence"] and row["has_error_guided_evidence"]:
        return "dual_supported_overlap"
    elif row["has_error_guided_evidence"]:
        return "error_guided_supported"
    elif row["has_clinical_evidence"]:
        return "clinical_supported"
    else:
        return "accepted_but_source_unclear"

source_audit["discovery_evidence"] = source_audit.apply(evidence_label, axis=1)

print("\nDiscovery evidence counts:")
print(source_audit["discovery_evidence"].value_counts())

display(
    source_audit[
        [
            "slice_name",
            "discovery_evidence",
            "nb8_harm",
            "priority_score",
            "val_logloss_lift",
            "val_recall_gap",
            "val_fn_lift"
        ]
    ].sort_values("nb8_harm", ascending=False)
)

# ------------------------------------------------------------
# 4. Define four discovery groups
# ------------------------------------------------------------

clinical_discovery_slices = sorted(clinical_supported)

error_guided_discovery_slices = sorted(error_supported)

dual_supported_slices = sorted(dual_supported)

dual_source_combined_slices = sorted(
    set(clinical_discovery_slices) | set(error_guided_discovery_slices)
)

print("\nFour discovery groups")
print("Clinical discovery:", len(clinical_discovery_slices))
print("Error-guided discovery:", len(error_guided_discovery_slices))
print("Dual-supported overlap:", len(dual_supported_slices))
print("Dual-source combined:", len(dual_source_combined_slices))

print("\nError-guided supported slices:")
print(error_guided_discovery_slices)

print("\nDual-supported slices:")
print(dual_supported_slices)

# ------------------------------------------------------------
# 5. Define governed positive-harm repair eligibility
# ------------------------------------------------------------

repair_registry = accepted_registry.copy()

# Clean identifiers.
repair_registry["slice_name"] = (
    repair_registry["slice_name"]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# Normalize final_accept robustly
# ------------------------------------------------------------

TRUE_VALUES = {
    "true",
    "1",
    "1.0",
    "yes",
    "y",
    "accepted",
    "accept",
}

FALSE_VALUES = {
    "false",
    "0",
    "0.0",
    "no",
    "n",
    "rejected",
    "reject",
    "",
    "nan",
    "none",
}


def normalize_acceptance(value):
    """
    Convert common registry acceptance representations
    into a Boolean value.
    """
    cleaned = str(value).strip().lower()

    if cleaned in TRUE_VALUES:
        return True

    if cleaned in FALSE_VALUES:
        return False

    raise ValueError(
        "Unrecognised final_accept value: "
        f"{value!r}"
    )


repair_registry["final_accept_bool"] = (
    repair_registry["final_accept"]
    .apply(normalize_acceptance)
)

# ------------------------------------------------------------
# Normalize artifact status
# ------------------------------------------------------------

repair_registry["artifact_clean"] = (
    repair_registry["artifact_flag"]
    .fillna("none")
    .astype(str)
    .str.lower()
    .str.strip()
)

# Treat only explicit artifact exclusions as ineligible.
artifact_exclusion_labels = {
    "pure_artifact",
    "mostly_missingness_artifact",
}

repair_registry["artifact_eligible"] = (
    ~repair_registry["artifact_clean"].isin(
        artifact_exclusion_labels
    )
)

# ------------------------------------------------------------
# Attach current Notebook 8 harm
# ------------------------------------------------------------

repair_registry["nb8_harm"] = (
    repair_registry["slice_name"]
    .map(slice_harm)
)

repair_registry["positive_harm"] = (
    repair_registry["nb8_harm"]
    .fillna(0)
    .gt(0)
)

# ------------------------------------------------------------
# Ensure row-level membership exists
# ------------------------------------------------------------

membership_names = set(
    slice_membership.columns
    .astype(str)
    .str.strip()
)

repair_registry["membership_available"] = (
    repair_registry["slice_name"]
    .isin(membership_names)
)

# ------------------------------------------------------------
# Final repair eligibility
#
# Do not use exact slice_type_manual strings here.
# Clinical plausibility/type governance is already represented
# by final_accept, while artifact risk is handled explicitly.
# ------------------------------------------------------------

repair_registry["repair_accept"] = (
    repair_registry["final_accept_bool"]
    & repair_registry["artifact_eligible"]
    & repair_registry["positive_harm"]
    & repair_registry["membership_available"]
)

eligible_registry_names = set(
    repair_registry.loc[
        repair_registry["repair_accept"],
        "slice_name",
    ]
)

repair_eligible_combined_slices = sorted(
    set(dual_source_combined_slices)
    .intersection(
        eligible_registry_names
    )
)

# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

repair_eligibility_audit_columns = [
    "slice_name",
    "final_accept",
    "final_accept_bool",
    "artifact_flag",
    "artifact_clean",
    "artifact_eligible",
    "slice_type_manual",
    "nb8_harm",
    "positive_harm",
    "membership_available",
    "repair_accept",
]

repair_eligibility_audit_columns = [
    column
    for column in repair_eligibility_audit_columns
    if column in repair_registry.columns
]

repair_eligibility_audit = (
    repair_registry[
        repair_eligibility_audit_columns
    ]
    .sort_values(
        [
            "repair_accept",
            "nb8_harm",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    repair_eligibility_audit
)

print(
    "Final accepted:",
    int(
        repair_registry[
            "final_accept_bool"
        ].sum()
    ),
)

print(
    "Artifact eligible:",
    int(
        repair_registry[
            "artifact_eligible"
        ].sum()
    ),
)

print(
    "Positive current harm:",
    int(
        repair_registry[
            "positive_harm"
        ].sum()
    ),
)

print(
    "Membership available:",
    int(
        repair_registry[
            "membership_available"
        ].sum()
    ),
)

print(
    "Repair eligible:",
    int(
        repair_registry[
            "repair_accept"
        ].sum()
    ),
)

print(
    "\nRepair-eligible combined slices:",
    len(
        repair_eligible_combined_slices
    ),
)

print(
    repair_eligible_combined_slices
)

if not repair_eligible_combined_slices:
    raise ValueError(
        "The repair-eligible combined slice list is "
        "still empty. Inspect repair_eligibility_audit "
        "to identify which governance condition is "
        "excluding the slices."
    )
# ------------------------------------------------------------
# 5B. HIGH-CONFIDENCE COMBINED DISCOVERY
# Curated fusion:
# - dual-supported slices OR
# - sufficiently high positive harm
# ------------------------------------------------------------

HIGH_HARM_THRESHOLD = 0.08

high_confidence_combined_slices = sorted([
    s for s in repair_eligible_combined_slices
    if (
        (
            s in dual_supported_slices
        )
        or (
            s in slice_harm.index
            and slice_harm.loc[s] >= HIGH_HARM_THRESHOLD
        )
    )
])

print("\nHigh-confidence combined slices:", len(high_confidence_combined_slices))
print(high_confidence_combined_slices)
monitoring_only_slices = sorted(
    set(dual_source_combined_slices)
    - set(repair_eligible_combined_slices)
)

print(
    "\nMonitoring-only / non-repair slices:",
    len(monitoring_only_slices),
)

print(
    monitoring_only_slices
)

print("\nRepair-eligible combined slices:", len(repair_eligible_combined_slices))
print(repair_eligible_combined_slices)

print("\nMonitoring-only / non-positive harm slices:", len(monitoring_only_slices))
print(monitoring_only_slices)

# ------------------------------------------------------------
# 6. Summarise discovery arms
# ------------------------------------------------------------

def summarize_slice_arm(arm_name, slice_names):
    temp = source_audit[source_audit["slice_name"].isin(slice_names)].copy()

    existing = [s for s in slice_names if s in slice_membership.columns]

    if len(existing) > 0:
        covered = slice_membership[existing].astype(bool).any(axis=1)
        patients_covered = int(covered.sum())
        pct_patients_covered = 100 * covered.mean()
    else:
        patients_covered = 0
        pct_patients_covered = 0

    if len(temp) == 0:
        return {
            "arm": arm_name,
            "n_slices": 0,
            "mean_nb8_harm": np.nan,
            "max_nb8_harm": np.nan,
            "n_positive_harm_slices": 0,
            "mean_priority_score": np.nan,
            "mean_val_logloss_lift": np.nan,
            "mean_val_recall_gap": np.nan,
            "mean_val_fn_lift": np.nan,
            "patients_covered": patients_covered,
            "pct_patients_covered": pct_patients_covered,
        }

    return {
        "arm": arm_name,
        "n_slices": temp["slice_name"].nunique(),
        "mean_nb8_harm": temp["nb8_harm"].mean(),
        "max_nb8_harm": temp["nb8_harm"].max(),
        "n_positive_harm_slices": (temp["nb8_harm"] > 0).sum(),
        "mean_priority_score": temp["priority_score"].mean(),
        "mean_val_logloss_lift": temp["val_logloss_lift"].mean(),
        "mean_val_recall_gap": temp["val_recall_gap"].mean(),
        "mean_val_fn_lift": temp["val_fn_lift"].mean(),
        "patients_covered": patients_covered,
        "pct_patients_covered": pct_patients_covered,
    }

#dual_source_ablation_summary = pd.DataFrame([
#    summarize_slice_arm("clinical_discovery", clinical_discovery_slices),
#    summarize_slice_arm("error_guided_discovery", error_guided_discovery_slices),
#    summarize_slice_arm("dual_supported_overlap", dual_supported_slices),
#    summarize_slice_arm("dual_source_combined", dual_source_combined_slices),
#    summarize_slice_arm("repair_eligible_combined", repair_eligible_combined_slices),
#])
dual_source_ablation_summary = pd.DataFrame([
    summarize_slice_arm("clinical_discovery", clinical_discovery_slices),

    summarize_slice_arm("error_guided_discovery", error_guided_discovery_slices),

    summarize_slice_arm("dual_supported_overlap", dual_supported_slices),

    summarize_slice_arm("dual_source_combined", dual_source_combined_slices),

    summarize_slice_arm("repair_eligible_combined", repair_eligible_combined_slices),

    summarize_slice_arm("high_confidence_combined", high_confidence_combined_slices),
])

dual_source_ablation_summary["discovery_strength_score"] = (
    dual_source_ablation_summary["mean_nb8_harm"].fillna(0)
    + dual_source_ablation_summary["mean_priority_score"].fillna(0)
    + dual_source_ablation_summary["mean_val_logloss_lift"].fillna(0)
    + dual_source_ablation_summary["mean_val_fn_lift"].fillna(0)
)

dual_source_ablation_summary = dual_source_ablation_summary.sort_values(
    "discovery_strength_score",
    ascending=False
).reset_index(drop=True)

display(dual_source_ablation_summary)

# ------------------------------------------------------------
# 7. Build source-aware weight table for Notebook 9
# ------------------------------------------------------------

source_weight_table = source_audit[
    [
        "slice_name",
        "nb8_harm",
        "priority_score",
        "has_clinical_evidence",
        "has_error_guided_evidence",
        "has_dual_evidence",
        "discovery_evidence"
    ]
].copy()

positive_h = source_weight_table["nb8_harm"].clip(lower=0)

if positive_h.max() > 0:
    source_weight_table["harm_norm_for_repair"] = positive_h / positive_h.max()
else:
    source_weight_table["harm_norm_for_repair"] = 0.0

source_weight_table["harm_norm_for_repair"] = source_weight_table["harm_norm_for_repair"].clip(lower=0.05)

source_weight_table["combined_evidence_multiplier"] = np.where(
    source_weight_table["has_dual_evidence"],
    1.50,
    1.00
)

display(source_weight_table.sort_values("harm_norm_for_repair", ascending=False))

# ------------------------------------------------------------
# 8. Save Notebook 8 outputs for Notebook 9
# ------------------------------------------------------------

source_audit.to_csv("nb8_slice_source_audit.csv", index=False)
dual_source_ablation_summary.to_csv("nb8_dual_source_ablation_summary.csv", index=False)
source_weight_table.to_csv("nb8_source_weight_table.csv", index=False)
###################################################################################
## BASiC-Slice Ranker: Provenance-Aware Harm Prioritization
# ============================================================
# BASiC-SLICE RANKER
# Provenance-aware, harm-aware, artifact-aware slice prioritization
# ============================================================

rank_table = source_weight_table.copy()

# ------------------------------------------------------------
# 1. Harm score: positive harm only, normalized to [0, 1]
# ------------------------------------------------------------

rank_table["harm_positive"] = rank_table["nb8_harm"].clip(lower=0)

if rank_table["harm_positive"].max() > 0:
    rank_table["harm_norm"] = (
        rank_table["harm_positive"] / rank_table["harm_positive"].max()
    )
else:
    rank_table["harm_norm"] = 0.0

# ------------------------------------------------------------
# 2. Dual-source evidence
# ------------------------------------------------------------

rank_table["dual_source_evidence"] = rank_table["has_dual_evidence"].astype(int)

# ------------------------------------------------------------
# 3. Clinical plausibility
# ------------------------------------------------------------

rank_table["clinical_plausibility"] = rank_table["has_clinical_evidence"].astype(int)

# ------------------------------------------------------------
# 4. Stability score
# Use registry pass_persistence if available
# ------------------------------------------------------------

if "pass_persistence" in source_audit.columns:
    persistence_map = source_audit.set_index("slice_name")["pass_persistence"].astype(bool).astype(int)
    rank_table["stability_score"] = rank_table["slice_name"].map(persistence_map).fillna(0)
else:
    rank_table["stability_score"] = 1.0

# ------------------------------------------------------------
# 5. Support score
# Normalize patient coverage per slice
# ------------------------------------------------------------

support_values = {}

for s in rank_table["slice_name"]:
    if s in slice_membership.columns:
        support_values[s] = slice_membership[s].astype(bool).sum()
    else:
        support_values[s] = 0

rank_table["support_n"] = rank_table["slice_name"].map(support_values)

if rank_table["support_n"].max() > 0:
    rank_table["support_score"] = rank_table["support_n"] / rank_table["support_n"].max()
else:
    rank_table["support_score"] = 0.0

# ------------------------------------------------------------
# 6. Artifact risk
# ------------------------------------------------------------

artifact_map = source_audit.set_index("slice_name")["artifact_flag"].astype(str).str.lower()

rank_table["artifact_flag_clean"] = rank_table["slice_name"].map(artifact_map).fillna("none")

rank_table["artifact_risk"] = rank_table["artifact_flag_clean"].isin([
    "pure_artifact",
    "mostly_missingness_artifact",
    "possible_data_corruption"
]).astype(int)

# ------------------------------------------------------------
# 7. Redundancy risk
# ------------------------------------------------------------

redundancy_map = source_audit.set_index("slice_name")["redundancy_flag"].astype(str).str.lower()

rank_table["redundancy_flag_clean"] = rank_table["slice_name"].map(redundancy_map).fillna("unique")

rank_table["redundancy_risk"] = rank_table["redundancy_flag_clean"].str.contains(
    "duplicate|nested|near_duplicate",
    regex=True
).astype(int)

# ------------------------------------------------------------
# 8. Final BASiC-Slice rank score
# ------------------------------------------------------------

rank_table["basic_slice_rank_score"] = (
    0.35 * rank_table["harm_norm"]
    + 0.20 * rank_table["dual_source_evidence"]
    + 0.15 * rank_table["clinical_plausibility"]
    + 0.15 * rank_table["stability_score"]
    + 0.10 * rank_table["support_score"]
    - 0.25 * rank_table["artifact_risk"]
    - 0.10 * rank_table["redundancy_risk"]
)

rank_table = rank_table.sort_values(
    "basic_slice_rank_score",
    ascending=False
).reset_index(drop=True)

display(
    rank_table[
        [
            "slice_name",
            "basic_slice_rank_score",
            "nb8_harm",
            "harm_norm",
            "dual_source_evidence",
            "clinical_plausibility",
            "stability_score",
            "support_score",
            "artifact_risk",
            "redundancy_risk",
            "discovery_evidence"
        ]
    ]
)

rank_table.to_csv("nb8_basic_slice_rank_table.csv", index=False)

print("Saved: nb8_basic_slice_rank_table.csv")
# ============================================================
# SELECT TOP-RANKED REPAIR SLICES
# ============================================================

TOP_K = 7

ranked_repair_candidates = rank_table[
    (rank_table["nb8_harm"] > 0)
    & (rank_table["artifact_risk"] == 0)
].copy()

top_ranked_repair_slices = ranked_repair_candidates.head(TOP_K)["slice_name"].tolist()

print("Top-ranked repair slices:", len(top_ranked_repair_slices))
print(top_ranked_repair_slices)

pd.Series(top_ranked_repair_slices, name="slice_name").to_csv(
    "nb8_top_ranked_repair_slices.csv",
    index=False
)

print("Saved: nb8_top_ranked_repair_slices.csv")
###################################################################################

pd.Series(clinical_discovery_slices, name="slice_name").to_csv(
    "nb8_clinical_discovery_slices.csv",
    index=False
)

pd.Series(error_guided_discovery_slices, name="slice_name").to_csv(
    "nb8_error_guided_discovery_slices.csv",
    index=False
)

pd.Series(dual_supported_slices, name="slice_name").to_csv(
    "nb8_dual_supported_slices.csv",
    index=False
)

pd.Series(dual_source_combined_slices, name="slice_name").to_csv(
    "nb8_dual_source_combined_slices.csv",
    index=False
)
if not repair_eligible_combined_slices:
    raise ValueError(
        "Refusing to save an empty repair-eligible "
        "combined slice handoff."
    )

invalid_repair_slices = [
    slice_name
    for slice_name in repair_eligible_combined_slices
    if slice_name not in slice_membership.columns
]

if invalid_repair_slices:
    raise ValueError(
        "Repair-eligible slices missing from membership: "
        f"{invalid_repair_slices}"
    )
pd.Series(
    repair_eligible_combined_slices,
    name="slice_name",
).to_csv(
    "nb8_repair_eligible_combined_slices.csv",
    index=False,
)
###############
pd.Series(high_confidence_combined_slices, name="slice_name").to_csv(
    "nb8_high_confidence_combined_slices.csv",
    index=False
)
###############
pd.Series(monitoring_only_slices, name="slice_name").to_csv(
    "nb8_monitoring_only_slices.csv",
    index=False
)

print("\nSaved:")
print("nb8_slice_source_audit.csv")
print("nb8_dual_source_ablation_summary.csv")
print("nb8_source_weight_table.csv")
print("nb8_clinical_discovery_slices.csv")
print("nb8_error_guided_discovery_slices.csv")
print("nb8_dual_supported_slices.csv")
print("nb8_dual_source_combined_slices.csv")
print("nb8_repair_eligible_combined_slices.csv")
print("nb8_monitoring_only_slices.csv")
##################
print("nb8_high_confidence_combined_slices.csv")

In [ ]:
# ============================================================
# REPAIR REGISTRY CATEGORY DIAGNOSTIC
# Run only after repair_registry has been constructed
# ============================================================

required_diagnostic_columns = [
    "slice_type_manual",
    "artifact_flag",
    "final_accept",
]

missing_diagnostic_columns = [
    column
    for column in required_diagnostic_columns
    if column not in repair_registry.columns
]

if missing_diagnostic_columns:
    raise KeyError(
        "Repair registry is missing diagnostic columns: "
        f"{missing_diagnostic_columns}"
    )

print(
    "Unique slice_type_manual values:"
)

print(
    repair_registry[
        "slice_type_manual"
    ]
    .fillna("<missing>")
    .astype(str)
    .str.lower()
    .str.strip()
    .value_counts(
        dropna=False
    )
)

print(
    "\nUnique artifact_flag values:"
)

print(
    repair_registry[
        "artifact_flag"
    ]
    .fillna("<missing>")
    .astype(str)
    .str.lower()
    .str.strip()
    .value_counts(
        dropna=False
    )
)

print(
    "\nUnique final_accept values:"
)

print(
    repair_registry[
        "final_accept"
    ]
    .fillna("<missing>")
    .astype(str)
    .str.lower()
    .str.strip()
    .value_counts(
        dropna=False
    )
)

print(
    "\nRepair-eligibility outcome:"
)

print(
    repair_registry[
        "repair_accept"
    ]
    .value_counts(
        dropna=False
    )
)

In [ ]:
check_combined = pd.read_csv(
    "nb8_repair_eligible_combined_slices.csv"
)

display(check_combined)

print(
    "Number of repair-eligible combined slices:",
    len(check_combined)
)

In [ ]:
check_weights = pd.read_csv(
    "nb8_source_weight_table.csv"
)

display(
    check_weights[
        check_weights["slice_name"].isin(
            check_combined["slice_name"]
        )
    ]
)

In [ ]:
# ============================================================
# DIAGNOSE WHY ONLY 1 REGISTRY SLICE IS AVAILABLE
# ============================================================

registry_names = set(full_registry["slice_name"].astype(str))
membership_names = set(slice_membership.columns.astype(str))

full_registry["final_accept_clean"] = full_registry["final_accept"].astype(str).str.lower().str.strip()

print("Registry slices:", len(registry_names))
print("Slice membership columns:", len(membership_names))
print("Accepted in registry:", (full_registry["final_accept_clean"] == "yes").sum())

accepted_all = full_registry[full_registry["final_accept_clean"] == "yes"].copy()

accepted_all["exists_in_slice_membership"] = accepted_all["slice_name"].isin(membership_names)

display(
    accepted_all[
        [
            "slice_name",
            "final_accept",
            "exists_in_slice_membership",
            "manual_plausibility",
            "artifact_flag",
            "redundancy_flag",
            "slice_type_manual",
            "priority_score"
        ]
    ].sort_values("exists_in_slice_membership")
)

missing_accepted = accepted_all.loc[
    ~accepted_all["exists_in_slice_membership"],
    "slice_name"
].tolist()

print("\nAccepted but missing from slice_membership:")
print(missing_accepted)

In [ ]:
# ============================================================
# CHECK HIGH-PRIORITY REJECTED SLICES
# ============================================================

display(
    full_registry[
        [
            "slice_name",
            "final_accept",
            "manual_plausibility",
            "artifact_flag",
            "redundancy_flag",
            "slice_type_manual",
            "priority_score",
            "pass_support",
            "pass_harm",
            "pass_persistence"
        ]
    ].sort_values("priority_score", ascending=False).head(15)
)

In [ ]:
## ============================================================
## CREATE REPAIR VS MONITORING ELIGIBILITY
## ============================================================
#
#reg = full_registry.copy()
#
#reg["final_accept_clean"] = reg["final_accept"].astype(str).str.lower().str.strip()
#reg["artifact_clean"] = reg["artifact_flag"].astype(str).str.lower().str.strip()
#reg["slice_type_clean"] = reg["slice_type_manual"].astype(str).str.lower().str.strip()
#
#reg["repair_accept"] = (
#    (reg["final_accept_clean"] == "yes") &
#    (~reg["artifact_clean"].isin(["pure_artifact", "mostly_missingness_artifact"])) &
#    (reg["slice_type_clean"].isin([
#        "clinical_harm_slice",
#        "operational_harm_slice",
#        "mixed_harm_slice"
#    ]))
#)
#
#reg["monitoring_accept"] = (
#    reg["artifact_clean"].isin([
#        "mostly_missingness_artifact",
#        "possible_documentation_artifact",
#        "operational_artifact"
#    ])
#)
#
#display(
#    reg[
#        [
#            "slice_name",
#            "final_accept",
#            "repair_accept",
#            "monitoring_accept",
#            "artifact_flag",
#            "slice_type_manual",
#            "priority_score"
#        ]
#    ].sort_values("priority_score", ascending=False)
#)

In [ ]:
# ============================================================
# CORRECT NON-REPAIR / MONITORING SET
# Includes positive-harm artifact slices too
# ============================================================

non_repair_combined_slices = sorted(
    set(dual_source_combined_slices) - set(repair_eligible_combined_slices)
)

print("Non-repair / monitoring combined slices:", len(non_repair_combined_slices))
print(non_repair_combined_slices)

pd.Series(non_repair_combined_slices, name="slice_name").to_csv(
    "nb8_non_repair_monitoring_slices.csv",
    index=False
)

***************************************************END*********************************************************************

# New ranker Ek = H + D + S. Start**************************************************

In [ ]:
# ============================================================
# REMOVE STALE REVISED OUTPUTS BEFORE CLEAN RERUN
# ============================================================

from pathlib import Path

revised_nb8_outputs = [
    "nb8_asap_complete_candidate_audit.csv",
    "nb8_asap_candidate_pool_with_ablations.csv",
    "nb8_asap_guardrail_ablation_summary.csv",
    "nb8_asap_guardrail_newly_admitted_detail.csv",
    "nb8_asap_constrained_basic_rank_table.csv",
    "nb8_asap_constrained_basic_top10.csv",
    "nb8_asap_guardrail_ablation_top10.csv",
    "nb8_asap_constrained_weight_grid_summary.csv",
    "nb8_asap_constrained_weight_grid_rankings.csv",
    "nb8_asap_constrained_weight_grid_diagnostics.csv",
]

for filename in revised_nb8_outputs:
    path = Path(filename)

    if path.exists():
        path.unlink()

        print(
            "Removed stale output:",
            filename
        )

In [ ]:
t_runtime_asap = start_timer(
    "Complete ASAP governance"
)
# ============================================================
# REVISED FRAMEWORK — STAGE 1
# FREEZE ASAP AS THE ELIGIBILITY/GOVERNANCE LAYER
# ============================================================

import os
import ast
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Start from the COMPLETE candidate/registry pool
# ------------------------------------------------------------

asap_pool = full_registry.copy()

if "slice_name" not in asap_pool.columns:
    raise KeyError("full_registry must contain a 'slice_name' column.")

# ============================================================
# COMPLETE CANDIDATE MEMBERSHIP FOR GUARDRAIL ABLATIONS
# ============================================================

# Preserve the accepted/canonical membership matrix.
candidate_membership = slice_membership.copy()

# ------------------------------------------------------------
# 1. Canonical aliases from error-guided discovery
# ------------------------------------------------------------

# ------------------------------------------------------------
# Canonical aliases from candidate discovery
# ------------------------------------------------------------

CANDIDATE_ALIAS_MAP = {
    "err_hr_missing_and_abdominal_pain":
        "abdominal_pain_and_hr_missing",

    "err_ckd_and_spo2_missing":
        "ckd_and_spo2_missing",

    "err_unclassified_and_missingness":
        "unclassified_and_vitals_missing_ge_3",

    "err_vitals_missing_ge_3":
        "vitals_missing_ge_3",
}

# ------------------------------------------------------------
# 2. Direct binary candidates available in df
# ------------------------------------------------------------

DIRECT_BINARY_CANDIDATES = [
    "abdominal_pain",
    "chest_pain",
    "back_pain",
    "cough",
    "headache",
    "ckd",
    "dm",
    "htn",
    "chd",
    "asthma",
    "pneumonia",
    "hr_missing",
    "spo2_missing",
    "arrival_mode_unknown",
    "arrival_mode_ambulance",
    "unclassified",
]

for candidate_name in DIRECT_BINARY_CANDIDATES:
    if (
        candidate_name in df.columns
        and candidate_name not in candidate_membership.columns
    ):
        candidate_membership[candidate_name] = (
            pd.to_numeric(
                df[candidate_name],
                errors="coerce"
            )
            .fillna(0)
            .gt(0)
            .astype(int)
        )
if (
    "arrival_mode_ambulance"
    not in candidate_membership.columns
):
    if "arrival_mode_ambulance" in df.columns:
        candidate_membership[
            "arrival_mode_ambulance"
        ] = (
            pd.to_numeric(
                df["arrival_mode_ambulance"],
                errors="coerce"
            )
            .fillna(0)
            .gt(0)
            .astype(int)
        )

    elif "arrival_mode" in df.columns:
        candidate_membership[
            "arrival_mode_ambulance"
        ] = (
            df["arrival_mode"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
            .eq("ambulance")
            .astype(int)
        )

    else:
        print(
            "Warning: arrival_mode_ambulance "
            "could not be reconstructed."
        )
# ------------------------------------------------------------
# 3. Reconstruct simple derived candidates when available
# ------------------------------------------------------------

if (
    "vitals_missing_count" in df.columns
    and "vitals_missing_ge_3"
    not in candidate_membership.columns
):
    candidate_membership["vitals_missing_ge_3"] = (
        pd.to_numeric(
            df["vitals_missing_count"],
            errors="coerce"
        )
        .fillna(0)
        .ge(3)
        .astype(int)
    )
# ------------------------------------------------------------
# Reconstruct the exact error-guided intersection:
# SpO2 missing AND unclassified complaint
# ------------------------------------------------------------

if (
    "err_spo2_missing_and_unclassified"
    not in candidate_membership.columns
):
    if (
        "spo2_missing" in candidate_membership.columns
        and "unclassified" in candidate_membership.columns
    ):
        candidate_membership[
            "err_spo2_missing_and_unclassified"
        ] = (
            candidate_membership[
                "spo2_missing"
            ].astype(bool)
            &
            candidate_membership[
                "unclassified"
            ].astype(bool)
        ).astype(int)

    elif (
        "spo2_missing" in df.columns
        and "unclassified" in df.columns
    ):
        candidate_membership[
            "err_spo2_missing_and_unclassified"
        ] = (
            pd.to_numeric(
                df["spo2_missing"],
                errors="coerce"
            )
            .fillna(0)
            .gt(0)
            &
            pd.to_numeric(
                df["unclassified"],
                errors="coerce"
            )
            .fillna(0)
            .gt(0)
        ).astype(int)

    else:
        print(
            "Warning: err_spo2_missing_and_unclassified "
            "could not be reconstructed."
        )
# ------------------------------------------------------------
# 4. Map each registry candidate to a membership definition
# ------------------------------------------------------------

asap_pool["membership_name"] = (
    asap_pool["slice_name"]
    .astype(str)
    .map(
        lambda name: CANDIDATE_ALIAS_MAP.get(
            name,
            name
        )
    )
)
if asap_pool["slice_name"].duplicated().any():
    duplicated_slice_names = (
        asap_pool.loc[
            asap_pool["slice_name"].duplicated(
                keep=False
            ),
            "slice_name"
        ]
        .astype(str)
        .tolist()
    )

    raise ValueError(
        "Duplicate candidate names exist in full_registry: "
        f"{sorted(set(duplicated_slice_names))}"
    )
asap_pool["exists_in_membership"] = (
    asap_pool["membership_name"]
    .isin(candidate_membership.columns)
)

# Audit candidates whose membership still cannot be reconstructed.
unresolved_membership_candidates = (
    asap_pool.loc[
        ~asap_pool["exists_in_membership"],
        [
            "slice_name",
            "membership_name",
            "final_accept",
            "artifact_flag",
            "redundancy_flag",
        ]
    ]
    .copy()
)

print(
    "Complete candidate-pool rows:",
    len(asap_pool)
)

print(
    "Candidates with reconstructable membership:",
    int(asap_pool["exists_in_membership"].sum())
)

print(
    "Candidates without reconstructable membership:",
    int((~asap_pool["exists_in_membership"]).sum())
)

if len(unresolved_membership_candidates):
    display(unresolved_membership_candidates)

# ------------------------------------------------------------
# 2. Standardize text fields
# ------------------------------------------------------------

def clean_text_series(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

for col in [
    "final_accept",
    "artifact_flag",
    "redundancy_flag",
    "slice_type_manual",
    "manual_plausibility",
]:
    if col in asap_pool.columns:
        asap_pool[f"{col}_clean"] = clean_text_series(asap_pool[col])

# ------------------------------------------------------------
# 3. Convert diverse registry representations to binary flags
# ------------------------------------------------------------

TRUE_TEXT = {
    "1", "true", "yes", "y", "pass", "passed", "accept",
    "accepted", "plausible", "stable", "replicated"
}

FALSE_TEXT = {
    "0", "false", "no", "n", "fail", "failed", "reject",
    "rejected", "implausible", "unstable", "not replicated"
}

def to_binary_flag(series, default=np.nan):
    def convert(value):
        if pd.isna(value):
            return default

        if isinstance(value, (bool, np.bool_)):
            return int(value)

        if isinstance(value, (int, float, np.integer, np.floating)):
            if np.isnan(value):
                return default
            return int(value > 0)

        text = str(value).strip().lower()

        if text in TRUE_TEXT:
            return 1

        if text in FALSE_TEXT:
            return 0

        # Numeric strings
        try:
            return int(float(text) > 0)
        except ValueError:
            return default

    return series.apply(convert)

# Support
if "pass_support" in asap_pool.columns:
    asap_pool["asap_support_pass"] = to_binary_flag(
        asap_pool["pass_support"], default=0
    )
else:
    # Fallback: use registry final support where available.
    support_col = next(
        (
            c for c in [
                "n_train",
                "train_n",
                "support",
                "slice_support",
                "n"
            ]
            if c in asap_pool.columns
        ),
        None
    )

    if support_col is None:
        raise KeyError(
            "Could not identify pass_support or a support-count column."
        )

    MIN_SUPPORT = 50
    asap_pool["asap_support_pass"] = (
        pd.to_numeric(
            asap_pool[support_col],
            errors="coerce"
        ).fillna(0) >= MIN_SUPPORT
    ).astype(int)

# ------------------------------------------------------------
# Current repair-harm requirement
# Use Notebook 8 harm, not only the historical registry flag.
# ------------------------------------------------------------

current_harm_lookup = (
    pd.Series(slice_harm)
    .astype(float)
    .to_dict()
)

# ============================================================
# CURRENT REPAIR-HARM REQUIREMENT
# Map harm through the canonical membership definition.
# ============================================================

current_harm_lookup = (
    pd.Series(slice_harm)
    .astype(float)
    .to_dict()
)

# Candidate aliases inherit the harm of the patient subgroup
# represented by membership_name.
asap_pool["nb8_harm"] = (
    asap_pool["membership_name"]
    .map(current_harm_lookup)
)

# Retain direct-name harm only for auditing.
asap_pool["nb8_harm_direct_name"] = (
    asap_pool["slice_name"]
    .map(current_harm_lookup)
)

HARM_EPSILON = 1e-12

asap_pool["asap_harm_pass"] = (
    asap_pool["nb8_harm"]
    .fillna(-np.inf)
    .gt(HARM_EPSILON)
    .astype(int)
)

# Preserve the historical registry field separately.
if "pass_harm" in asap_pool.columns:
    asap_pool["registry_pass_harm"] = (
        to_binary_flag(
            asap_pool["pass_harm"],
            default=0
        )
    )

# Persistence / replication
if "pass_persistence" in asap_pool.columns:
    asap_pool["asap_persistence_pass"] = to_binary_flag(
        asap_pool["pass_persistence"], default=0
    )
else:
    persistence_candidates = [
        c for c in [
            "replicated",
            "heldout_replication",
            "temporal_persistence",
            "validation_pass"
        ]
        if c in asap_pool.columns
    ]

    if persistence_candidates:
        asap_pool["asap_persistence_pass"] = to_binary_flag(
            asap_pool[persistence_candidates[0]],
            default=0
        )
    else:
        # Do not silently assume pass if the information does not exist.
        asap_pool["asap_persistence_pass"] = np.nan

# ============================================================
# REPLICATION / STABILITY GOVERNANCE
# ============================================================

stability_candidates = [
    column
    for column in [
        "stability_score",
        "bootstrap_stability",
        "stability",
        "pass_stability",
    ]
    if column in asap_pool.columns
]

HAS_INDEPENDENT_STABILITY = bool(stability_candidates)

if HAS_INDEPENDENT_STABILITY:
    stability_col = stability_candidates[0]

    if "pass" in stability_col.lower():
        asap_pool["asap_stability_pass"] = (
            to_binary_flag(
                asap_pool[stability_col],
                default=0
            )
        )

        asap_pool["asap_stability_value"] = (
            asap_pool["asap_stability_pass"]
            .astype(float)
        )

    else:
        asap_pool["asap_stability_value"] = (
            pd.to_numeric(
                asap_pool[stability_col],
                errors="coerce"
            )
        )

        STABILITY_THRESHOLD = 0.60

        asap_pool["asap_stability_pass"] = (
            asap_pool["asap_stability_value"]
            .ge(STABILITY_THRESHOLD)
            .fillna(False)
            .astype(int)
        )

    print(
        "Independent stability criterion detected:",
        stability_col
    )

else:
    # Do not pretend stability is an independent criterion.
    asap_pool["asap_stability_value"] = np.nan

    # Combined replication/stability decision.
    asap_pool["asap_stability_pass"] = (
        asap_pool["asap_persistence_pass"]
        .copy()
    )

    print(
        "No independent stability measure was found. "
        "Persistence and stability will be treated as one "
        "combined replication/stability guardrail."
    )

# ------------------------------------------------------------
# Clinical plausibility
# Registry values: high, moderate, low
# ------------------------------------------------------------

if "manual_plausibility" not in asap_pool.columns:
    raise KeyError(
        "manual_plausibility is required for ASAP governance."
    )

plausibility_text = (
    asap_pool["manual_plausibility"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

PLAUSIBILITY_SCORE_MAP = {
    "high": 1.0,
    "moderate": 0.5,
    "medium": 0.5,
    "low": 0.0,

    "yes": 1.0,
    "plausible": 1.0,
    "clinically plausible": 1.0,
    "operationally plausible": 1.0,

    "no": 0.0,
    "implausible": 0.0,
}

asap_pool["asap_plausibility_score"] = (
    plausibility_text
    .map(PLAUSIBILITY_SCORE_MAP)
)

unknown_plausibility = asap_pool.loc[
    asap_pool["asap_plausibility_score"].isna(),
    ["slice_name", "manual_plausibility"]
]

if len(unknown_plausibility):
    display(unknown_plausibility)

    raise ValueError(
        "Unrecognized manual_plausibility values were found. "
        "Update PLAUSIBILITY_SCORE_MAP before proceeding."
    )

PLAUSIBILITY_THRESHOLD = 0.5

asap_pool["asap_plausibility_pass"] = (
    asap_pool["asap_plausibility_score"]
    >= PLAUSIBILITY_THRESHOLD
).astype(int)

# ============================================================
# ARTIFACT GOVERNANCE
# ============================================================

if "artifact_flag" not in asap_pool.columns:
    raise KeyError(
        "artifact_flag is required for ASAP governance."
    )

# Standardize spelling and separators.
artifact_text = (
    asap_pool["artifact_flag"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

# Correct a known registry typo.
artifact_text = artifact_text.replace({
    "possibe": "possible",
})

asap_pool["artifact_flag_clean"] = artifact_text

ARTIFACT_SCORE_MAP = {
    # No meaningful artifact concern
    "unlikely": 0.0,
    "unlikely_artifact": 0.0,
    "none": 0.0,
    "no": 0.0,
    "false": 0.0,
    "0": 0.0,

    # Possible artifact:
    # eligible, but monitored
    "possible": 0.5,
    "possible_artifact": 0.5,
    "possible_missingness_artifact": 0.5,
    "possible_workflow_artifact": 0.5,

    # Likely or confirmed artifact:
    # hard ASAP veto
    "likely": 1.0,
    "likely_artifact": 1.0,
    "mostly_missingness_artifact": 1.0,
    "pure_artifact": 1.0,
    "artifact": 1.0,
    "confirmed_artifact": 1.0,
    "high_artifact_risk": 1.0,
    "yes": 1.0,
    "true": 1.0,
    "1": 1.0,
}

asap_pool["artifact_risk_score"] = (
    asap_pool["artifact_flag_clean"]
    .map(ARTIFACT_SCORE_MAP)
)

unknown_artifact_labels = (
    asap_pool.loc[
        asap_pool["artifact_risk_score"].isna(),
        [
            "slice_name",
            "artifact_flag",
            "artifact_flag_clean",
        ]
    ]
    .drop_duplicates()
    .copy()
)

if len(unknown_artifact_labels):
    display(unknown_artifact_labels)

    raise ValueError(
        "Unrecognised artifact labels remain. "
        "Inspect artifact_flag_clean and update "
        "ARTIFACT_SCORE_MAP."
    )

asap_pool["asap_artifact_veto"] = (
    asap_pool["artifact_risk_score"]
    .ge(1.0)
    .astype(int)
)

asap_pool["asap_artifact_pass"] = (
    1 - asap_pool["asap_artifact_veto"]
)

asap_pool["artifact_monitoring_flag"] = (
    asap_pool["artifact_risk_score"]
    .eq(0.5)
    .astype(int)
)

print("\nArtifact governance summary:")

display(
    asap_pool[
        [
            "artifact_flag_clean",
            "artifact_risk_score",
            "asap_artifact_pass",
            "artifact_monitoring_flag",
        ]
    ]
    .value_counts()
    .rename("count")
    .reset_index()
)

# Redundancy veto
if "redundancy_flag" not in asap_pool.columns:
    raise KeyError("redundancy_flag is required.")

redundancy_text = clean_text_series(
    asap_pool["redundancy_flag"]
)

redundancy_reject_labels = {
    "redundant",
    "duplicate",
    "nested_duplicate",
    "alias",
    "yes",
    "1",
    "true",
}

asap_pool["asap_redundancy_veto"] = (
    redundancy_text.isin(redundancy_reject_labels)
).astype(int)

asap_pool["asap_redundancy_pass"] = (
    1 - asap_pool["asap_redundancy_veto"]
)

# ------------------------------------------------------------
# 4. Freeze the full ASAP eligibility rule
# ------------------------------------------------------------

required_asap_pass_columns = [
    "asap_support_pass",
    "asap_harm_pass",
    "asap_persistence_pass",
    "asap_plausibility_pass",
    "asap_artifact_pass",
    "asap_redundancy_pass",
]

# Include stability separately only when it is independently measured.
if HAS_INDEPENDENT_STABILITY:
    required_asap_pass_columns.append(
        "asap_stability_pass"
    )

asap_pool["asap_eligible_full"] = (
    asap_pool[required_asap_pass_columns]
    .fillna(0)
    .astype(int)
    .all(axis=1)
    &
    asap_pool["exists_in_membership"]
)

# Retain the original registry decision for audit, not as the new rule.
if "final_accept" in asap_pool.columns:
    asap_pool["registry_final_accept_yes"] = (
        clean_text_series(asap_pool["final_accept"]) == "yes"
    ).astype(int)

positive_harm_series = (
    pd.Series(slice_harm)
    .clip(lower=0)
    .astype(float)
)

maximum_positive_harm = max(
    positive_harm_series.max(),
    1e-12
)

canonical_harm_norm_lookup = (
    positive_harm_series
    / maximum_positive_harm
).to_dict()

asap_pool["harm_norm"] = (
    asap_pool["membership_name"]
    .map(canonical_harm_norm_lookup)
    .fillna(0.0)
)

# Candidate aliases inherit the provenance status of the
# canonical patient subgroup they represent.
asap_pool["dual_source_evidence"] = (
    asap_pool["membership_name"]
    .isin(dual_supported_slices)
    .astype(int)
)

display_cols = [
    "slice_name",
    "exists_in_membership",
    *required_asap_pass_columns,
    "asap_eligible_full",
    "nb8_harm",
    "dual_source_evidence",
]

display(
    asap_pool[display_cols]
    .sort_values(
        ["asap_eligible_full", "nb8_harm"],
        ascending=[False, False]
    )
)

asap_pool.to_csv(
    "nb8_asap_complete_candidate_audit.csv",
    index=False
)

print(
    "Full ASAP-eligible slices:",
    int(asap_pool["asap_eligible_full"].sum())
)
print("Saved: nb8_asap_complete_candidate_audit.csv")
NB8_REVISED_ASAP_READY = True
stop_timer(
    "Complete ASAP governance",
    t_runtime_asap
)

In [ ]:
print(
    asap_pool[
        [
            "slice_name",
            "membership_name",
            "nb8_harm",
            "asap_harm_pass",
            "artifact_flag_clean",
            "asap_artifact_pass",
            "asap_redundancy_pass",
            "asap_eligible_full",
        ]
    ]
    .sort_values(
        "nb8_harm",
        ascending=False
    )
    .to_string(index=False)
)

In [ ]:
if not globals().get(
    "NB8_REVISED_ASAP_READY",
    False
):
    raise RuntimeError(
        "Run the corrected full-ASAP screening "
        "cell before this validation cell."
    )
n_candidate_rows = len(asap_pool)

n_reconstructable = int(
    asap_pool[
        "exists_in_membership"
    ].sum()
)

print(
    f"Candidate membership reconstruction: "
    f"{n_reconstructable}/{n_candidate_rows}"
)

if n_reconstructable < n_candidate_rows:
    unresolved_names = (
        asap_pool.loc[
            ~asap_pool[
                "exists_in_membership"
            ],
            [
                "slice_name",
                "membership_name",
            ]
        ]
    )

    display(unresolved_names)

    raise ValueError(
        "Not all candidate memberships were reconstructed. "
        "Resolve the displayed candidates before running "
        "guardrail ablations."
    )

In [ ]:
if not globals().get(
    "NB8_REVISED_ASAP_READY",
    False
):
    raise RuntimeError(
        "Run the corrected full-ASAP screening "
        "cell before guardrail ablation."
    )
required_columns_after_asap = [
    "slice_name",
    "membership_name",
    "exists_in_membership",
    "asap_support_pass",
    "asap_harm_pass",
    "asap_persistence_pass",
    "asap_plausibility_pass",
    "artifact_risk_score",
    "asap_artifact_pass",
    "asap_redundancy_pass",
    "asap_eligible_full",
    "nb8_harm",
    "harm_norm",
    "dual_source_evidence",
]

missing_columns_after_asap = [
    column
    for column in required_columns_after_asap
    if column not in asap_pool.columns
]

if missing_columns_after_asap:
    raise KeyError(
        "ASAP screening did not create: "
        f"{missing_columns_after_asap}"
    )

print("ASAP screening columns are complete.")

In [ ]:
t_runtime_guardrail = start_timer(
    "ASAP one-at-a-time guardrail ablation"
)
# ============================================================
# EXPERIMENT 1
# ASAP GUARDRAIL ABLATION ON THE COMPLETE CANDIDATE POOL
# ============================================================

from collections import OrderedDict
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Define the active governance rules
# ------------------------------------------------------------

BASE_GOVERNANCE_RULES = OrderedDict({
    "support": "asap_support_pass",
    "harm": "asap_harm_pass",
    "replication": "asap_persistence_pass",
    "plausibility": "asap_plausibility_pass",
    "artifact": "asap_artifact_pass",
    "redundancy": "asap_redundancy_pass",
})

# Add stability separately only when it is genuinely independent.
if HAS_INDEPENDENT_STABILITY:
    BASE_GOVERNANCE_RULES[
        "stability"
    ] = "asap_stability_pass"

# ------------------------------------------------------------
# 2. Define one-at-a-time guardrail ablations
# ------------------------------------------------------------

GUARDRAIL_ABLATIONS = OrderedDict({
    "full_ASAP": None,

    "without_plausibility":
        "plausibility",

    "without_replication_stability":
        (
            "stability"
            if HAS_INDEPENDENT_STABILITY
            else "replication"
        ),

    "without_artifact_veto":
        "artifact",

    "without_redundancy_control":
        "redundancy",
})

# ------------------------------------------------------------
# 3. Eligibility function
# ------------------------------------------------------------

def evaluate_asap_configuration(
    pool,
    removed_rule=None,
):
    active_columns = [
        column
        for rule_name, column
        in BASE_GOVERNANCE_RULES.items()
        if rule_name != removed_rule
    ]

    missing_columns = [
        column
        for column in active_columns
        if column not in pool.columns
    ]

    if missing_columns:
        raise KeyError(
            "Missing governance columns: "
            f"{missing_columns}"
        )

    eligibility = (
        pool[active_columns]
        .fillna(0)
        .astype(int)
        .all(axis=1)
        &
        pool["exists_in_membership"]
        .fillna(False)
        .astype(bool)
    )

    return eligibility.astype(int)

# ------------------------------------------------------------
# 4. Create eligibility columns
# ------------------------------------------------------------

for configuration, removed_rule in (
    GUARDRAIL_ABLATIONS.items()
):
    eligibility_column = (
        f"eligible__{configuration}"
    )

    asap_pool[eligibility_column] = (
        evaluate_asap_configuration(
            pool=asap_pool,
            removed_rule=removed_rule,
        )
    )

# ------------------------------------------------------------
# 5. Compare each configuration with full ASAP
# ------------------------------------------------------------

guardrail_ablation_rows = []
newly_admitted_detail_frames = []

full_eligible_set = set(
    asap_pool.loc[
        asap_pool[
            "eligible__full_ASAP"
        ].eq(1),
        "slice_name"
    ]
    .astype(str)
)

for configuration, removed_rule in (
    GUARDRAIL_ABLATIONS.items()
):
    eligibility_column = (
        f"eligible__{configuration}"
    )

    current = (
        asap_pool.loc[
            asap_pool[
                eligibility_column
            ].eq(1)
        ]
        .copy()
    )

    current_set = set(
        current["slice_name"]
        .astype(str)
    )

    newly_admitted = (
        current_set - full_eligible_set
    )

    # --------------------------------------------------------
    # Detailed rows newly admitted by an ablation
    # --------------------------------------------------------

    newly_admitted_detail = (
        current.loc[
            current["slice_name"]
            .astype(str)
            .isin(newly_admitted),
            [
                "slice_name",
                "membership_name",
                "asap_support_pass",
                "asap_harm_pass",
                "asap_persistence_pass",
                "asap_stability_pass",
                "asap_plausibility_pass",
                "artifact_flag_clean",
                "artifact_risk_score",
                "asap_artifact_pass",
                "asap_redundancy_pass",
                "nb8_harm",
                "harm_norm",
                "dual_source_evidence",
            ]
        ]
        .copy()
    )

    if len(newly_admitted_detail):
        newly_admitted_detail[
            "configuration"
        ] = configuration

        newly_admitted_detail[
            "removed_rule"
        ] = (
            removed_rule
            if removed_rule is not None
            else "none"
        )

        newly_admitted_detail_frames.append(
            newly_admitted_detail
        )

    # --------------------------------------------------------
    # Contamination counts
    # --------------------------------------------------------

    n_implausible = int(
        (
            current[
                "asap_plausibility_pass"
            ] == 0
        ).sum()
    )

    n_replication_failure = int(
        (
            current[
                "asap_persistence_pass"
            ] == 0
        ).sum()
    )

    n_stability_failure = int(
        (
            current[
                "asap_stability_pass"
            ] == 0
        ).sum()
    )

    n_artifact = int(
        (
            current[
                "asap_artifact_pass"
            ] == 0
        ).sum()
    )

    n_redundant = int(
        (
            current[
                "asap_redundancy_pass"
            ] == 0
        ).sum()
    )

    # --------------------------------------------------------
    # Patient coverage using canonical memberships
    # --------------------------------------------------------

    membership_names = (
        current["membership_name"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    valid_memberships = [
        membership_name
        for membership_name
        in membership_names
        if membership_name
        in candidate_membership.columns
    ]

    if valid_memberships:
        covered = (
            candidate_membership[
                valid_memberships
            ]
            .astype(bool)
            .any(axis=1)
        )

        patients_covered = int(
            covered.sum()
        )

        pct_patients_covered = float(
            100 * covered.mean()
        )

    else:
        patients_covered = 0
        pct_patients_covered = 0.0

    positive_harm_values = (
        current.loc[
            current["nb8_harm"] > 0,
            "nb8_harm"
        ]
    )

    guardrail_ablation_rows.append({
        "configuration":
            configuration,

        "removed_rule":
            (
                removed_rule
                if removed_rule is not None
                else "none"
            ),

        "n_eligible_slices":
            int(len(current)),

        "n_newly_admitted_vs_full_ASAP":
            int(len(newly_admitted)),

        "newly_admitted_slices":
            sorted(newly_admitted),

        "n_implausible_in_eligible_set":
            n_implausible,

        "n_replication_failures_in_eligible_set":
            n_replication_failure,

        "n_stability_failures_in_eligible_set":
            n_stability_failure,

        "n_artifact_in_eligible_set":
            n_artifact,

        "n_redundant_in_eligible_set":
            n_redundant,

        "patients_covered":
            patients_covered,

        "pct_patients_covered":
            pct_patients_covered,

        "mean_positive_harm":
            (
                float(
                    positive_harm_values.mean()
                )
                if len(positive_harm_values)
                else np.nan
            ),

        "max_harm":
            (
                float(
                    current["nb8_harm"].max()
                )
                if len(current)
                else np.nan
            ),
    })

# ------------------------------------------------------------
# 6. Build and save outputs
# ------------------------------------------------------------

guardrail_ablation_summary = pd.DataFrame(
    guardrail_ablation_rows
)

if newly_admitted_detail_frames:
    guardrail_newly_admitted_detail = (
        pd.concat(
            newly_admitted_detail_frames,
            ignore_index=True
        )
    )
else:
    guardrail_newly_admitted_detail = (
        pd.DataFrame(
            columns=[
                "configuration",
                "removed_rule",
                "slice_name",
                "membership_name",
            ]
        )
    )

display(
    guardrail_ablation_summary
)

display(
    guardrail_newly_admitted_detail
)

guardrail_ablation_summary.to_csv(
    "nb8_asap_guardrail_ablation_summary.csv",
    index=False
)

guardrail_newly_admitted_detail.to_csv(
    "nb8_asap_guardrail_newly_admitted_detail.csv",
    index=False
)

asap_pool.to_csv(
    "nb8_asap_candidate_pool_with_ablations.csv",
    index=False
)

print("Saved:")
print("- nb8_asap_guardrail_ablation_summary.csv")
print("- nb8_asap_guardrail_newly_admitted_detail.csv")
print("- nb8_asap_candidate_pool_with_ablations.csv")

In [ ]:
expected_ablation_columns = [
    "eligible__full_ASAP",
    "eligible__without_plausibility",
    "eligible__without_replication_stability",
    "eligible__without_artifact_veto",
    "eligible__without_redundancy_control",
]

missing_ablation_columns = [
    column
    for column in expected_ablation_columns
    if column not in asap_pool.columns
]

if missing_ablation_columns:
    raise KeyError(
        "Guardrail ablation columns are missing: "
        f"{missing_ablation_columns}"
    )

print("All guardrail-ablation eligibility columns exist.")

display(
    guardrail_ablation_summary[
        [
            "configuration",
            "removed_rule",
            "n_eligible_slices",
            "n_newly_admitted_vs_full_ASAP",
            "n_implausible_in_eligible_set",
            "n_replication_failures_in_eligible_set",
            "n_artifact_in_eligible_set",
            "n_redundant_in_eligible_set",
        ]
    ]
)
stop_timer(
    "ASAP one-at-a-time guardrail ablation",
    t_runtime_guardrail
)

In [ ]:
t_runtime_ranker = start_timer(
    "BASiC repair-influence ranking"
)
# ============================================================
# REVISED FRAMEWORK — STAGE 2
# ASAP-CONSTRAINED BASiC PRIORITIZATION
# ============================================================

# ------------------------------------------------------------
# 1. Eligible pool
# ------------------------------------------------------------

asap_eligible_rank_pool = asap_pool.loc[
    asap_pool["asap_eligible_full"].eq(1)
    &
    asap_pool["nb8_harm"].gt(0)
    &
    asap_pool["harm_norm"].gt(0)
].copy()

if len(asap_eligible_rank_pool) == 0:
    raise ValueError(
        "No slices passed the full ASAP eligibility rule. "
        "Inspect nb8_asap_complete_candidate_audit.csv."
    )

# ------------------------------------------------------------
# 2. Build normalized support score
# ------------------------------------------------------------

# Prefer an actual support-count column.
support_count_candidates = [
    c for c in [
        "n_train",
        "train_n",
        "support",
        "slice_support",
        "n",
        "train_support"
    ]
    if c in asap_eligible_rank_pool.columns
]

if support_count_candidates:
    support_count_col = support_count_candidates[0]

    support_raw = pd.to_numeric(
        asap_eligible_rank_pool[support_count_col],
        errors="coerce"
    ).fillna(0)
else:
    # Recalculate directly from slice_membership.
    support_raw = (
        asap_eligible_rank_pool["slice_name"]
        .map({
            s: int(slice_membership[s].sum())
            for s in slice_membership.columns
        })
        .fillna(0)
    )

asap_eligible_rank_pool["support_count_ranker"] = (
    support_raw.astype(float)
)

max_support = max(
    asap_eligible_rank_pool[
        "support_count_ranker"
    ].max(),
    1.0
)

# Min-max or max scaling can be used.
# Use max scaling here because zero remains zero.
asap_eligible_rank_pool["support_norm_ranker"] = (
    asap_eligible_rank_pool[
        "support_count_ranker"
    ] / max_support
)

# ------------------------------------------------------------
# 3. Freeze starting weights
# ------------------------------------------------------------

PRIMARY_WEIGHTS = {
    "harm_norm": 0.60,
    "dual_source_evidence": 0.30,
    "support_norm_ranker": 0.10,
}

assert np.isclose(
    sum(PRIMARY_WEIGHTS.values()),
    1.0
), "Primary positive weights must sum to 1."

asap_eligible_rank_pool[
    "asap_constrained_basic_score"
] = (
    PRIMARY_WEIGHTS["harm_norm"]
    * asap_eligible_rank_pool["harm_norm"]
    +
    PRIMARY_WEIGHTS["dual_source_evidence"]
    * asap_eligible_rank_pool["dual_source_evidence"]
    +
    PRIMARY_WEIGHTS["support_norm_ranker"]
    * asap_eligible_rank_pool["support_norm_ranker"]
)

# ------------------------------------------------------------
# 4. Preserve original integrated rank score as comparator
# ------------------------------------------------------------

legacy_rank_table = pd.read_csv(
    "nb8_basic_slice_rank_table.csv"
)

legacy_score_col = next(
    (
        c for c in [
            "basic_slice_rank_score",
            "BASiC_Rank",
            "rank_score"
        ]
        if c in legacy_rank_table.columns
    ),
    None
)

legacy_cols = ["slice_name"]

if legacy_score_col:
    legacy_cols.append(legacy_score_col)

legacy_compare = legacy_rank_table[
    legacy_cols
].copy()

if legacy_score_col:
    legacy_compare = legacy_compare.rename(
        columns={
            legacy_score_col:
            "legacy_integrated_basic_score"
        }
    )

asap_constrained_rank_table = (
    asap_eligible_rank_pool
    .merge(
        legacy_compare,
        on="slice_name",
        how="left"
    )
    .sort_values(
        "asap_constrained_basic_score",
        ascending=False
    )
    .reset_index(drop=True)
)
# ============================================================
# VALIDATE THAT PROVENANCE CAN BE TESTED
# ============================================================

provenance_distribution = (
    asap_eligible_rank_pool[
        "dual_source_evidence"
    ]
    .value_counts(dropna=False)
    .sort_index()
)

print("Provenance distribution among ASAP-eligible slices:")
print(provenance_distribution)

n_dual_eligible = int(
    asap_eligible_rank_pool[
        "dual_source_evidence"
    ].sum()
)

eligible_dual_slices = (
    asap_eligible_rank_pool.loc[
        asap_eligible_rank_pool[
            "dual_source_evidence"
        ].eq(1),
        "slice_name"
    ]
    .astype(str)
    .tolist()
)

print(
    "\nDual-supported ASAP-eligible slices:",
    eligible_dual_slices
)

if n_dual_eligible == 0:
    raise ValueError(
        "The provenance-placement experiment cannot be "
        "identified because dual_source_evidence has no "
        "variation in the eligible pool."
    )
asap_constrained_rank_table[
    "asap_constrained_rank"
] = (
    np.arange(len(asap_constrained_rank_table)) + 1
)

REQUESTED_ASAP_TOP_K = 10

EFFECTIVE_ASAP_TOP_K = min(
    REQUESTED_ASAP_TOP_K,
    len(asap_constrained_rank_table)
)

if (
    EFFECTIVE_ASAP_TOP_K
    < REQUESTED_ASAP_TOP_K
):
    print(
        f"Only {EFFECTIVE_ASAP_TOP_K} positive-harm "
        "ASAP-eligible slices are available. "
        f"Using TOP{EFFECTIVE_ASAP_TOP_K}."
    )

asap_constrained_top10 = (
    asap_constrained_rank_table
    .head(EFFECTIVE_ASAP_TOP_K)
    ["slice_name"]
    .astype(str)
    .tolist()
)

display(
    asap_constrained_rank_table[
        [
            "asap_constrained_rank",
            "slice_name",
            "harm_norm",
            "dual_source_evidence",
            "support_count_ranker",
            "support_norm_ranker",
            "asap_constrained_basic_score",
            *(
                ["legacy_integrated_basic_score"]
                if "legacy_integrated_basic_score"
                in asap_constrained_rank_table.columns
                else []
            ),
        ]
    ]
)

print("Official revised TOP10:")
print(asap_constrained_top10)

asap_constrained_rank_table.to_csv(
    "nb8_asap_constrained_basic_rank_table.csv",
    index=False
)

pd.Series(
    asap_constrained_top10,
    name="slice_name"
).to_csv(
    "nb8_asap_constrained_basic_top10.csv",
    index=False
)

print("Saved:")
print("- nb8_asap_constrained_basic_rank_table.csv")
print("- nb8_asap_constrained_basic_top10.csv")
stop_timer(
    "BASiC repair-influence ranking",
    t_runtime_ranker
)

In [ ]:
print(
    asap_constrained_rank_table[
        [
            "slice_name",
            "nb8_harm",
            "harm_norm",
            "dual_source_evidence",
            "support_norm_ranker",
            "asap_constrained_basic_score",
        ]
    ]
)

print(
    "\nNumber of ranked slices:",
    len(asap_constrained_rank_table)
)

print(
    "\nDual-supported ranked slices:",
    asap_constrained_rank_table.loc[
        asap_constrained_rank_table[
            "dual_source_evidence"
        ].eq(1),
        "slice_name"
    ].tolist()
)

In [ ]:
t_runtime_weights = start_timer(
    "Repair-influence sensitivity analysis - 39 schemes"
)
# ============================================================
# EXPERIMENT 2
# SYSTEMATIC WEIGHT-GRID SENSITIVITY
# ============================================================

from itertools import product
from scipy.stats import spearmanr

REFERENCE_SCORE_COL = "asap_constrained_basic_score"

reference_ranking = (
    asap_constrained_rank_table
    .sort_values(
        REFERENCE_SCORE_COL,
        ascending=False
    )
    ["slice_name"]
    .astype(str)
    .tolist()
)

reference_top_sets = {
    k: set(reference_ranking[:k])
    for k in [5, 7, 10]
}

# Broad prespecified ranges.
harm_grid = np.round(
    np.arange(0.40, 0.81, 0.05),
    2
)

provenance_grid = np.round(
    np.arange(0.10, 0.51, 0.05),
    2
)

grid_results = []
grid_ranking_records = []

def jaccard_similarity(a, b):
    a = set(a)
    b = set(b)

    union = a | b

    if not union:
        return np.nan

    return len(a & b) / len(union)

reference_rank_map = {
    s: rank
    for rank, s in enumerate(
        reference_ranking,
        start=1
    )
}

scheme_counter = 0

for w_harm, w_provenance in product(
    harm_grid,
    provenance_grid
):
    w_support = round(
        1.0 - w_harm - w_provenance,
        10
    )

    # Keep only valid non-negative weight combinations.
    if w_support < -1e-9:
        continue

    if w_support > 0.20 + 1e-9:
        continue

    w_support = max(w_support, 0.0)

    scheme_counter += 1
    scheme_name = (
        f"grid_{scheme_counter:03d}"
        f"__H{w_harm:.2f}"
        f"__D{w_provenance:.2f}"
        f"__S{w_support:.2f}"
    )

    temp = asap_constrained_rank_table.copy()

    temp["grid_score"] = (
        w_harm * temp["harm_norm"]
        +
        w_provenance
        * temp["dual_source_evidence"]
        +
        w_support
        * temp["support_norm_ranker"]
    )

    temp = temp.sort_values(
        "grid_score",
        ascending=False
    ).reset_index(drop=True)

    temp["grid_rank"] = (
        np.arange(len(temp)) + 1
    )

    selected_top5 = (
        temp.head(5)["slice_name"]
        .astype(str)
        .tolist()
    )

    selected_top7 = (
        temp.head(7)["slice_name"]
        .astype(str)
        .tolist()
    )

    selected_top10 = (
        temp.head(10)["slice_name"]
        .astype(str)
        .tolist()
    )

    current_rank_map = dict(
        zip(
            temp["slice_name"].astype(str),
            temp["grid_rank"]
        )
    )

    common_slices = sorted(
        set(reference_rank_map)
        &
        set(current_rank_map)
    )

    if len(common_slices) >= 2:
        rank_correlation = spearmanr(
            [
                reference_rank_map[s]
                for s in common_slices
            ],
            [
                current_rank_map[s]
                for s in common_slices
            ]
        ).statistic
    else:
        rank_correlation = np.nan

    row = {
        "scheme": scheme_name,
        "w_harm": w_harm,
        "w_provenance": w_provenance,
        "w_support": w_support,

        "top5_membership": selected_top5,
        "top7_membership": selected_top7,
        "top10_membership": selected_top10,

        "top5_overlap_count": len(
            set(selected_top5)
            & reference_top_sets[5]
        ),
        "top7_overlap_count": len(
            set(selected_top7)
            & reference_top_sets[7]
        ),
        "top10_overlap_count": len(
            set(selected_top10)
            & reference_top_sets[10]
        ),

        "top5_jaccard": jaccard_similarity(
            selected_top5,
            reference_top_sets[5]
        ),
        "top7_jaccard": jaccard_similarity(
            selected_top7,
            reference_top_sets[7]
        ),
        "top10_jaccard": jaccard_similarity(
            selected_top10,
            reference_top_sets[10]
        ),

        "full_ranking_spearman": rank_correlation,
    }

    grid_results.append(row)

    grid_ranking_records.append(
        temp.assign(
            scheme=scheme_name,
            w_harm=w_harm,
            w_provenance=w_provenance,
            w_support=w_support,
        )[
            [
                "scheme",
                "w_harm",
                "w_provenance",
                "w_support",
                "slice_name",
                "grid_score",
                "grid_rank",
            ]
        ]
    )

basic_weight_grid_summary = pd.DataFrame(
    grid_results
)

basic_weight_grid_rankings = pd.concat(
    grid_ranking_records,
    ignore_index=True
)

display(
    basic_weight_grid_summary.sort_values(
        [
            "top10_overlap_count",
            "full_ranking_spearman"
        ],
        ascending=False
    )
)

basic_weight_grid_summary.to_csv(
    "nb8_asap_constrained_weight_grid_summary.csv",
    index=False
)

basic_weight_grid_rankings.to_csv(
    "nb8_asap_constrained_weight_grid_rankings.csv",
    index=False
)

print("Number of valid weight combinations:", len(
    basic_weight_grid_summary
))
print("Saved systematic weight-grid outputs.")

In [ ]:
# ============================================================
# WEIGHT-GRID ROBUSTNESS DIAGNOSTICS
# Distinguish membership sets from ranking orderings
# ============================================================

import ast
import numpy as np
import pandas as pd


def parse_membership(value):
    """
    Convert stored list/string membership into a clean tuple.
    The returned tuple preserves ranking order.
    """
    if isinstance(value, (list, tuple)):
        items = list(value)

    elif pd.isna(value):
        items = []

    else:
        try:
            parsed = ast.literal_eval(str(value))

            if isinstance(parsed, (list, tuple)):
                items = list(parsed)
            else:
                items = [parsed]

        except (ValueError, SyntaxError):
            items = [
                item.strip()
                for item in str(value).split(",")
                if item.strip()
            ]

    return tuple(
        str(item).strip()
        for item in items
        if str(item).strip()
    )


def canonical_membership_set(value):
    """
    Represent membership independently of ranking position.
    """
    ordered_membership = parse_membership(value)

    return tuple(
        sorted(
            set(ordered_membership)
        )
    )


def membership_ordering(value):
    """
    Preserve the exact ranking order.
    """
    return parse_membership(value)


diagnostic_grid = (
    basic_weight_grid_summary.copy()
)

# ------------------------------------------------------------
# Create separate set and ordering representations
# ------------------------------------------------------------

for k in [5, 7, 10]:
    source_col = f"top{k}_membership"

    diagnostic_grid[
        f"top{k}_membership_set"
    ] = diagnostic_grid[source_col].apply(
        canonical_membership_set
    )

    diagnostic_grid[
        f"top{k}_ordering"
    ] = diagnostic_grid[source_col].apply(
        membership_ordering
    )


# ------------------------------------------------------------
# Summarise robustness
# ------------------------------------------------------------

weight_grid_diagnostics = pd.DataFrame({
    "diagnostic": [
        "number_of_valid_weight_schemes",

        "number_of_unique_TOP5_membership_sets",
        "number_of_unique_TOP5_orderings",

        "number_of_unique_TOP7_membership_sets",
        "number_of_unique_TOP7_orderings",

        "number_of_unique_TOP10_membership_sets",
        "number_of_unique_TOP10_orderings",

        "minimum_TOP5_jaccard",
        "median_TOP5_jaccard",

        "minimum_TOP7_jaccard",
        "median_TOP7_jaccard",

        "minimum_TOP10_jaccard",
        "median_TOP10_jaccard",

        "minimum_full_ranking_spearman",
        "median_full_ranking_spearman",
    ],

    "value": [
        len(diagnostic_grid),

        diagnostic_grid[
            "top5_membership_set"
        ].nunique(),
        diagnostic_grid[
            "top5_ordering"
        ].nunique(),

        diagnostic_grid[
            "top7_membership_set"
        ].nunique(),
        diagnostic_grid[
            "top7_ordering"
        ].nunique(),

        diagnostic_grid[
            "top10_membership_set"
        ].nunique(),
        diagnostic_grid[
            "top10_ordering"
        ].nunique(),

        diagnostic_grid[
            "top5_jaccard"
        ].min(),
        diagnostic_grid[
            "top5_jaccard"
        ].median(),

        diagnostic_grid[
            "top7_jaccard"
        ].min(),
        diagnostic_grid[
            "top7_jaccard"
        ].median(),

        diagnostic_grid[
            "top10_jaccard"
        ].min(),
        diagnostic_grid[
            "top10_jaccard"
        ].median(),

        diagnostic_grid[
            "full_ranking_spearman"
        ].min(),
        diagnostic_grid[
            "full_ranking_spearman"
        ].median(),
    ]
})

display(
    weight_grid_diagnostics
)

weight_grid_diagnostics.to_csv(
    "nb8_asap_constrained_weight_grid_diagnostics.csv",
    index=False,
)

print(
    "Weight-grid diagnostics saved with membership "
    "sets separated from ranking orderings."
)
stop_timer(
    "Repair-influence sensitivity analysis - 39 schemes",
    t_runtime_weights
)

In [ ]:
# ============================================================
# CREATE POSITIVE-HARM TOP-K SETS FOR EACH ASAP ABLATION
# Used later in Notebook 9
# ============================================================

ablation_topk_records = []

REQUESTED_ABLATION_TOP_K = 10

for configuration in GUARDRAIL_ABLATIONS:
    eligible_col = (
        f"eligible__{configuration}"
    )

    temp = asap_pool.loc[
        asap_pool[eligible_col].eq(1)
        &
        asap_pool["nb8_harm"].gt(0)
        &
        asap_pool["harm_norm"].gt(0)
        &
        asap_pool["exists_in_membership"]
    ].copy()

    # Use canonical candidate memberships for support.
    support_lookup = {
        membership_name:
            int(
                candidate_membership[
                    membership_name
                ].sum()
            )
        for membership_name
        in candidate_membership.columns
    }

    temp["support_count_ranker"] = (
        temp["membership_name"]
        .map(support_lookup)
        .fillna(0)
        .astype(float)
    )

    max_support = max(
        temp[
            "support_count_ranker"
        ].max(),
        1.0
    )

    temp["support_norm_ranker"] = (
        temp["support_count_ranker"]
        / max_support
    )

    temp["ablation_rank_score"] = (
        0.60 * temp["harm_norm"]
        +
        0.30 * temp[
            "dual_source_evidence"
        ]
        +
        0.10 * temp[
            "support_norm_ranker"
        ]
    )

    # Preserve candidate identities, but prevent two aliases of the
    # same membership group from occupying multiple TOP-K positions.
    temp = (
        temp.sort_values(
            [
                "ablation_rank_score",
                "nb8_harm",
            ],
            ascending=False
        )
        .drop_duplicates(
            subset=["membership_name"],
            keep="first"
        )
        .reset_index(drop=True)
    )

    effective_top_k = min(
        REQUESTED_ABLATION_TOP_K,
        len(temp)
    )

    selected = temp.head(
        effective_top_k
    ).copy()

    for position, row in enumerate(
        selected.itertuples(
            index=False
        ),
        start=1
    ):
        ablation_topk_records.append({
            "configuration":
                configuration,

            "rank":
                position,

            "slice_name":
                row.slice_name,

            "membership_name":
                row.membership_name,

            "rank_score":
                float(
                    row.ablation_rank_score
                ),

            "nb8_harm":
                float(row.nb8_harm),

            "harm_norm":
                float(row.harm_norm),

            "dual_source_evidence":
                int(
                    row.dual_source_evidence
                ),

            "support_count_ranker":
                float(
                    row.support_count_ranker
                ),

            "requested_top_k":
                REQUESTED_ABLATION_TOP_K,

            "effective_top_k":
                effective_top_k,
        })

asap_ablation_top10_long = (
    pd.DataFrame(
        ablation_topk_records
    )
)

display(
    asap_ablation_top10_long
)

asap_ablation_top10_long.to_csv(
    "nb8_asap_guardrail_ablation_top10.csv",
    index=False
)

print(
    "\nNumber selected by configuration:"
)

display(
    asap_ablation_top10_long
    .groupby("configuration")
    .size()
    .rename("n_selected")
    .reset_index()
)

In [ ]:
# ============================================================
# FINAL INTEGRITY CHECKS
# ASAP-CONSTRAINED BASiC
# ============================================================
required_integrity_objects = [
    "asap_pool",
    "asap_constrained_rank_table",
    "asap_constrained_top10",
    "asap_ablation_top10_long",
]

missing_integrity_objects = [
    object_name
    for object_name
    in required_integrity_objects
    if object_name not in globals()
]

if missing_integrity_objects:
    raise NameError(
        "Integrity checks cannot run yet. "
        f"Missing objects: {missing_integrity_objects}"
    )
eligible = asap_pool.loc[
    asap_pool[
        "asap_eligible_full"
    ].eq(1)
].copy()

ranked = (
    asap_constrained_rank_table
    .copy()
)

# ------------------------------------------------------------
# Eligibility checks
# ------------------------------------------------------------

assert (
    eligible[
        "asap_plausibility_score"
    ] >= 0.5
).all(), (
    "A low-plausibility slice passed ASAP."
)

assert (
    eligible[
        "asap_artifact_pass"
    ] == 1
).all(), (
    "A hard-artifact slice passed ASAP."
)

assert (
    eligible[
        "asap_redundancy_pass"
    ] == 1
).all(), (
    "A redundant slice passed ASAP."
)

# ------------------------------------------------------------
# Ranking checks
# ------------------------------------------------------------

assert len(ranked) > 0, (
    "The revised ranking table is empty."
)

assert (
    ranked["nb8_harm"] > 0
).all(), (
    "A non-positive-harm slice entered "
    "the revised ranker."
)

assert (
    ranked["harm_norm"] > 0
).all(), (
    "A zero normalized-harm slice entered "
    "the revised ranker."
)

assert (
    ranked[
        "dual_source_evidence"
    ].nunique() >= 2
), (
    "Provenance does not vary in the "
    "eligible ranking pool."
)

# ------------------------------------------------------------
# Expected dual-supported slices
# ------------------------------------------------------------

expected_dual_supported = {
    "abdominal_pain_and_hr_missing",
    "ckd_and_spo2_missing",
    "unclassified_and_vitals_missing_ge_3",
}

observed_dual_supported = set(
    ranked.loc[
        ranked[
            "dual_source_evidence"
        ].eq(1),
        "slice_name"
    ]
)

missing_dual_supported = (
    expected_dual_supported
    -
    observed_dual_supported
)

assert not missing_dual_supported, (
    "Expected dual-supported slices were "
    f"excluded: {sorted(missing_dual_supported)}"
)

# ------------------------------------------------------------
# TOP-K checks
# ------------------------------------------------------------

assert (
    len(asap_constrained_top10)
    <= len(ranked)
)

assert set(
    asap_constrained_top10
).issubset(
    set(ranked["slice_name"])
)

# No zero-harm slices in any full-ASAP TOP-K output.
full_asap_topk = (
    asap_ablation_top10_long.loc[
        asap_ablation_top10_long[
            "configuration"
        ].eq("full_ASAP")
    ]
)

assert (
    full_asap_topk["nb8_harm"] > 0
).all(), (
    "A zero-harm slice entered the "
    "full-ASAP ablation TOP-K."
)

print(
    "All ASAP-constrained BASiC "
    "integrity checks passed."
)

In [ ]:
# ============================================================
# FINAL NOTEBOOK 8 OUTPUT VALIDATION
# ============================================================

from pathlib import Path

required_nb8_outputs = [
    "nb8_asap_complete_candidate_audit.csv",
    "nb8_asap_candidate_pool_with_ablations.csv",
    "nb8_asap_guardrail_ablation_summary.csv",
    "nb8_asap_guardrail_newly_admitted_detail.csv",
    "nb8_asap_constrained_basic_rank_table.csv",
    "nb8_asap_constrained_basic_top10.csv",
    "nb8_asap_guardrail_ablation_top10.csv",
    "nb8_asap_constrained_weight_grid_summary.csv",
    "nb8_asap_constrained_weight_grid_rankings.csv",
    "nb8_asap_constrained_weight_grid_diagnostics.csv",
]

missing_output_files = []

for filename in required_nb8_outputs:
    path = Path(filename)
    exists = path.exists()

    print(
        filename,
        "FOUND" if exists else "MISSING"
    )

    if not exists:
        missing_output_files.append(filename)

if missing_output_files:
    raise FileNotFoundError(
        "Notebook 8 did not create: "
        f"{missing_output_files}"
    )

print(
    "All revised Notebook 8 outputs were created. "
    "Notebook 9 may now be executed."
)

# New ranker Ek = H + D + S. End**************************************************

Identify worst slice

In [ ]:
# Identify worst harmful slice safely
positive_harm_for_worst = slice_harm.dropna()

if len(positive_harm_for_worst) == 0:
    raise ValueError("slice_harm is empty after dropping NaNs. Check slice_membership and harm computation.")

worst_slice = positive_harm_for_worst.idxmax()

print("Worst slice:", worst_slice)
print("Worst-slice harm:", positive_harm_for_worst.loc[worst_slice])

In [ ]:
# ============================================================
# WHY DID THE WORST SLICE CHANGE?
# Registry-priority worst vs Notebook-8 harm worst
# ============================================================
#
# arrival_mode_unknown was strongest in the Notebook 5 registry because
# the registry priority score emphasized support, persistence, and recall gap.
#
# abdominal_pain_and_hr_missing can become worst in Notebook 8 because
# Notebook 8 recomputes harm using:
#     oof_logloss lift + false-negative lift + high-confidence contradiction lift
#
# These are different definitions of "worst". Keep both visible.

if "priority_score" in full_registry.columns:
    print("Top slices by Notebook 5 registry priority:")
    display(
        full_registry.sort_values("priority_score", ascending=False)[
            ["slice_name", "final_accept", "priority_score", "val_logloss_lift",
             "val_recall_gap", "val_fn_lift", "artifact_flag", "redundancy_flag"]
        ].head(10)
    )

print("\nTop slices by Notebook 8 recomputed harm:")
display(
    slice_harm.dropna().sort_values(ascending=False).head(10).rename("nb8_harm").to_frame()
)

In [ ]:
#worst_slice = slice_harm.idxmax()
#
#print("Worst slice:", worst_slice)

Build new weights (CORE CONTRIBUTION)

In [ ]:
def build_weights(df, slice_membership, harm_norm, lambda_=2.0, worst_boost=2.0):
    weights = np.ones(len(df))

    for slice_name in slice_membership.columns:
        mask = slice_membership[slice_name] == 1

        base_weight = lambda_ * harm_norm.get(slice_name, 0)

        if slice_name == worst_slice:
            base_weight *= worst_boost

        weights[mask] += base_weight

    return weights

weights_slice = build_weights(df, slice_membership, harm_norm)

df["w_slice"] = weights_slice

df["w_slice"].describe()

Stronger suspicious-label downweighting

In [ ]:
def suspicious_downweight(df, rho=0.5):
    suspicious = (
        (df["oof_logloss"] > df["oof_logloss"].quantile(0.9)) |
        (df["high_conf_contradiction"] == 1)
    )

    weights = np.ones(len(df))
    weights[suspicious] = rho

    return weights

df["w_suspicious"] = suspicious_downweight(df)

Final weights

In [ ]:
df["w_final"] = df["w_slice"] * df["w_suspicious"]

df["w_final"].describe()

In [ ]:
import numpy as np
import pandas as pd

# Check prerequisites
print("slice_harm exists:", "slice_harm" in globals())
print("slice_membership exists:", "slice_membership" in globals())

# Build final harm-based sample weights
df["w_final"] = 1.0

positive_harm = slice_harm.dropna()
positive_harm = positive_harm[positive_harm > 0]

print("Number of positive-harm slices:", len(positive_harm))
print(positive_harm.sort_values(ascending=False).head(10))

# Normalize harm scores
if len(positive_harm) > 0:
    positive_harm_norm = positive_harm / positive_harm.max()
else:
    positive_harm_norm = positive_harm

lambda_harm = 1.0

for slice_name, harm_score in positive_harm_norm.items():
    if slice_name in slice_membership.columns:
        df["w_final"] += lambda_harm * harm_score * slice_membership[slice_name].values

# Prevent extreme weights
df["w_final"] = df["w_final"].clip(lower=1.0, upper=5.0)

print(df["w_final"].describe())
print("Upweighted patients:", (df["w_final"] > 1).sum())
print("Percent upweighted:", round((df["w_final"] > 1).mean() * 100, 2))

In [ ]:
print("w_final exists?", "w_final" in df.columns)
print(df["w_final"].describe())

In [ ]:
print(model)

Train model with new weights

In [ ]:
#import numpy as np
#import pandas as pd
#
#from sklearn.model_selection import StratifiedKFold
#from sklearn.compose import ColumnTransformer
#from sklearn.pipeline import Pipeline
#from sklearn.linear_model import LogisticRegression
#from sklearn.impute import SimpleImputer
#from sklearn.preprocessing import OneHotEncoder, StandardScaler
#
## -------------------------
## 1. Define columns safely
## -------------------------
#y_col = "y"
#
#exclude_cols = [
#    "y",
#    "disposition",
#    "arrival_hour_bin_raw",
#    "p_oof",
#    "oof_pred_proba",
#    "oof_logloss",
#    "oof_pred_label_05",
#    "oof_error_05",
#    "false_negative_05",
#    "high_conf_contradiction",
#    "p_harm_weighted",
#    "w_final",
#]
#
#categorical_cols = [
#    "arrival_mode",
#    "arrival_month",
#    "arrival_day",
#    "arrival_hour_bin",
#]
#
#numeric_cols = [
#    c for c in df.columns
#    if c not in exclude_cols
#    and c not in categorical_cols
#    and pd.api.types.is_numeric_dtype(df[c])
#]
#
#X_cols = numeric_cols + categorical_cols
#
#X = df[X_cols].copy()
#y = df[y_col].astype(int)
#w = df["w_final"].astype(float)
#
#print("Text columns handled as categorical:")
#print(X.select_dtypes(include=["object", "string"]).columns.tolist())
#
## -------------------------
## 2. Correct preprocessing
## -------------------------
#weighted_preprocess = ColumnTransformer(
#    transformers=[
#        ("num", Pipeline([
#            ("imputer", SimpleImputer(strategy="median")),
#            ("scaler", StandardScaler())
#        ]), numeric_cols),
#
#        ("cat", Pipeline([
#            ("imputer", SimpleImputer(strategy="most_frequent")),
#            ("onehot", OneHotEncoder(handle_unknown="ignore"))
#        ]), categorical_cols),
#    ]
#)
#
#weighted_model = Pipeline([
#    ("preprocess", weighted_preprocess),
#    ("lr", LogisticRegression(max_iter=5000, solver="lbfgs"))
#])
#
#print(weighted_model)
#print(weighted_model.named_steps.keys())
#
## -------------------------
## 3. Weighted OOF training
## -------------------------
#cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#oof = np.zeros(len(df), dtype=float)
#
#for fold, (tr, te) in enumerate(cv.split(X, y), 1):
#    weighted_model.fit(
#        X.iloc[tr],
#        y.iloc[tr],
#        lr__sample_weight=w.iloc[tr]
#    )
#
#    oof[te] = weighted_model.predict_proba(X.iloc[te])[:, 1]
#    print(f"Fold {fold} done")
#
#df["p_harm_weighted"] = np.clip(oof, 1e-6, 1 - 1e-6)
#
#print("Done.")
#print(df["p_harm_weighted"].head())

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# -------------------------
# 1. Define columns safely
# -------------------------
y_col = "y"

# Avoid leakage: remove targets, diagnostic predictions, and all repair weights.
base_exclude_cols = {
    "y",
    "disposition",
    "arrival_hour_bin_raw",
    "oof_pred_proba",
    "oof_logloss",
    "oof_pred_label_05",
    "oof_error_05",
    "false_negative_05",
    "high_conf_contradiction",
}

categorical_cols = [
    "arrival_mode",
    "arrival_month",
    "arrival_day",
    "arrival_hour_bin",
]

categorical_cols = [c for c in categorical_cols if c in df.columns]

numeric_cols = [
    c for c in df.columns
    if c not in base_exclude_cols
    and c not in categorical_cols
    and not c.startswith("p_")
    and not c.startswith("w_")
    and pd.api.types.is_numeric_dtype(df[c])
]

X_cols = numeric_cols + categorical_cols

X = df[X_cols].copy()
y = df[y_col].astype(int)
w = df["w_final"].astype(float)

print("Numeric cols:", len(numeric_cols))
print("Categorical cols:", categorical_cols)
print("Text columns handled as categorical:")
print(X.select_dtypes(include=["object", "string"]).columns.tolist())

# -------------------------
# 2. Correct preprocessing
# -------------------------
weighted_preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),
    ]
)

weighted_model = Pipeline([
    ("preprocess", weighted_preprocess),
    ("lr", LogisticRegression(max_iter=5000, solver="lbfgs"))
])

print(weighted_model)

# -------------------------
# 3. Weighted OOF training
# -------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(df), dtype=float)

for fold, (tr, te) in enumerate(cv.split(X, y), 1):
    weighted_model.fit(
        X.iloc[tr],
        y.iloc[tr],
        lr__sample_weight=w.iloc[tr]
    )

    oof[te] = weighted_model.predict_proba(X.iloc[te])[:, 1]
    print(f"Fold {fold} done")

df["p_harm_weighted"] = np.clip(oof, 1e-6, 1 - 1e-6)

print("Done.")
print(df["p_harm_weighted"].head())

Global metrics

In [ ]:
print("AUROC:", roc_auc_score(y, df["p_harm_weighted"]))
print("AUPRC:", average_precision_score(y, df["p_harm_weighted"]))

preds = (df["p_harm_weighted"] >= 0.5).astype(int)

print("Recall:", recall_score(y, preds))
print("Precision:", precision_score(y, preds))
print("Brier:", brier_score_loss(y, df["p_harm_weighted"]))

CRITICAL: Worst-slice evaluation

In [ ]:
mask = slice_membership[worst_slice] == 1

y_slice = y[mask]
p_slice = df.loc[mask, "p_harm_weighted"]

preds_slice = (p_slice >= 0.5).astype(int)

print("Worst slice:", worst_slice)
print("AUPRC:", average_precision_score(y_slice, p_slice))
print("Recall:", recall_score(y_slice, preds_slice))

In [ ]:
def compute_slice_metrics(df, slice_membership, prob_col, threshold=0.5):
    results = []

    for col in slice_membership.columns:
        mask = slice_membership[col] == 1

        if mask.sum() < 50:
            continue

        y_slice = df.loc[mask, "y"]
        p_slice = df.loc[mask, prob_col]

        preds = (p_slice >= threshold).astype(int)

        recall = recall_score(y_slice, preds)
        auprc = average_precision_score(y_slice, p_slice)

        results.append({
            "slice": col,
            "recall": recall,
            "auprc": auprc,
            "n": mask.sum()
        })

    return pd.DataFrame(results)

In [ ]:
slice_results = compute_slice_metrics(df, slice_membership, "p_harm_weighted")

mean_recall = slice_results["recall"].mean()
worst_recall = slice_results["recall"].min()

mean_auprc = slice_results["auprc"].mean()
worst_auprc = slice_results["auprc"].min()

print("Mean recall:", mean_recall)
print("Worst recall:", worst_recall)
print("avg_to_worst_recall_gap:", mean_recall - worst_recall)

print("Mean AUPRC:", mean_auprc)
print("Worst AUPRC:", worst_auprc)
print("avg_to_worst_auprc_gap:", mean_auprc - worst_auprc)

In [ ]:
required_objects = ["df", "slice_harm", "slice_membership"]

for obj in required_objects:
    print(obj, "exists:", obj in globals())

In [ ]:
# Save dataset with diagnostics
df.to_csv("df_with_diagnostics.csv", index=False)

# Save slice harm
slice_harm.to_csv("slice_harm.csv")

# Save slice membership
slice_membership.to_csv("slice_membership.csv", index=False)